# RHI Live Runtime v29 — Operational Equivalence Probe Gate

Δ **Purpose:** make slot agreement operational instead of lexical.

Core laws:

$$\boxed{\text{Slots agree only if they induce the same operational ranking.}}$$

$$\boxed{\text{The model slot must not accept what the compiler rejects.}}$$

Stack:

$$Q_{raw}\rightarrow C_{root}\rightarrow S_{model}\rightarrow \mathcal{C}_{probe}\rightarrow B(C_{root})\parallel B(S_{model})\rightarrow G_{equiv}\rightarrow C_Q\rightarrow \Psi/\Omega/\bot$$

The probe generator uses six layers: original/baseline, anti-fit instantiations, boundary-stress near-misses, operation paraphrases, preserved-function violations, and cross-domain distractors.

Outputs exactly two files:

```text
rhi_v29_<run_id>_bundle.json
rhi_v29_<run_id>_summary.csv
```


In [1]:

from __future__ import annotations
import os, re, sys, json, uuid, time, random, traceback, subprocess, importlib, hashlib
from dataclasses import dataclass, asdict, field
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter
import numpy as np
import pandas as pd

ROOT=Path.cwd(); OUT_DIR=ROOT/'rhi_v29_outputs'; OUT_DIR.mkdir(exist_ok=True)
RUN_ID='rhi_v29_'+uuid.uuid4().hex[:10]
SEED=29; random.seed(SEED); np.random.seed(SEED)
MODEL_ID_OR_PATH=os.environ.get('RHI_MODEL','Qwen/Qwen2.5-1.5B-Instruct')
AUTO_INSTALL_MISSING_DEPS=True; REQUIRE_MODEL_FOR_PSI=True
RUN_PROMPT_LIMIT=36; MAX_DEPTH=2
BRANCH_ROLES=['construct','verify','repair','counter']
MAX_NEW_TOKENS_PACKET=260; MAX_NEW_TOKENS_BRANCH=170; MAX_NEW_TOKENS_SHAPER=120
TEMPERATURE_PACKET=0.20; TEMPERATURE_BRANCH=0.40; TEMPERATURE_SHAPER=0.20
TAU_ACCEPT=0.75; CSDI_ACCEPT=0.90; R_Q_ACCEPT=0.40; R_Q_WARN=0.20; LAMBDA_CSDI=2.0
ROOT_REJECT_THRESHOLD=0.0; MODEL_ACCEPT_THRESHOLD=0.0
SHAPER_ENABLED=True
print('RHI v29 — Operational Equivalence Probe Gate')
print('RUN_ID:', RUN_ID)
print('MODEL:', MODEL_ID_OR_PATH)


RHI v29 — Operational Equivalence Probe Gate
RUN_ID: rhi_v29_0aa867fa0a
MODEL: Qwen/Qwen2.5-1.5B-Instruct


In [2]:

def _module_available(m):
    try: return importlib.util.find_spec(m) is not None
    except Exception: return False

def ensure_runtime_dependencies():
    status={'checked':True,'attempted_install':False,'missing_before':[],'missing_after':[],'errors':[]}
    req=[('torch','torch'),('transformers','transformers'),('sentencepiece','sentencepiece'),('google.protobuf','protobuf')]
    for mod,pip in req:
        if not _module_available(mod): status['missing_before'].append(pip)
    if status['missing_before'] and AUTO_INSTALL_MISSING_DEPS:
        status['attempted_install']=True
        try:
            subprocess.run([sys.executable,'-m','pip','install',*sorted(set(status['missing_before']))],check=True)
            importlib.invalidate_caches()
        except Exception as e: status['errors'].append(repr(e))
    for mod,pip in req:
        if not _module_available(mod): status['missing_after'].append(pip)
    return status

DEPENDENCY_STATUS=ensure_runtime_dependencies()
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def get_device_info():
    d={'torch_version':torch.__version__,'cuda_available':bool(torch.cuda.is_available()),'device_count':int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,'cuda_version':getattr(torch.version,'cuda',None)}
    if torch.cuda.is_available(): d['gpu_name']=torch.cuda.get_device_name(0)
    return d
DEVICE_INFO=get_device_info()

tokenizer=None; model=None; MODEL_READY=False; MODEL_GENERATION_READY=False; MODEL_ERROR=None; SMOKE_TEXT=None

def load_model():
    global tokenizer, model, MODEL_READY, MODEL_ERROR
    try:
        print('Loading model:', MODEL_ID_OR_PATH)
        tokenizer=AutoTokenizer.from_pretrained(MODEL_ID_OR_PATH)
        if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        model=AutoModelForCausalLM.from_pretrained(MODEL_ID_OR_PATH,dtype=dtype,device_map='auto' if torch.cuda.is_available() else None)
        if not torch.cuda.is_available(): model=model.to('cpu')
        model.eval(); MODEL_READY=True
        print('MODEL_READY:', MODEL_READY, 'DEVICE:', next(model.parameters()).device)
    except Exception:
        MODEL_READY=False; MODEL_ERROR=traceback.format_exc(); print(MODEL_ERROR)

def model_smoke_test():
    global MODEL_GENERATION_READY, SMOKE_TEXT, MODEL_ERROR
    if not MODEL_READY: return
    try:
        msg=[{'role':'user','content':'Reply with READY only.'}]
        rendered=tokenizer.apply_chat_template(msg,tokenize=False,add_generation_prompt=True)
        inp=tokenizer(rendered,return_tensors='pt').to(next(model.parameters()).device)
        with torch.no_grad(): out=model.generate(**inp,max_new_tokens=8,do_sample=False,return_dict_in_generate=True,pad_token_id=tokenizer.eos_token_id)
        gen=out.sequences[0][inp.input_ids.shape[1]:]
        SMOKE_TEXT=tokenizer.decode(gen,skip_special_tokens=True).strip(); MODEL_GENERATION_READY=bool(SMOKE_TEXT)
        print('MODEL_GENERATION_READY:', MODEL_GENERATION_READY, 'SMOKE:', SMOKE_TEXT)
    except Exception:
        MODEL_GENERATION_READY=False; MODEL_ERROR=traceback.format_exc(); print(MODEL_ERROR)

load_model(); model_smoke_test()
print('DEPENDENCY_STATUS:', DEPENDENCY_STATUS)
print('DEVICE_INFO:', DEVICE_INFO)


Loading model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


MODEL_READY: True DEVICE: cuda:0
MODEL_GENERATION_READY: True SMOKE: READY
DEPENDENCY_STATUS: {'checked': True, 'attempted_install': False, 'missing_before': [], 'missing_after': [], 'errors': []}
DEVICE_INFO: {'torch_version': '2.11.0+cu126', 'cuda_available': True, 'device_count': 1, 'cuda_version': '12.6', 'gpu_name': 'NVIDIA GeForce RTX 4060'}


In [3]:

class State(Enum): PSI='Ψ'; OMEGA='Ω'; BOTTOM='⊥'
@dataclass
class Slot:
    raw_input:str; profile:str; inferred_task:str; required_operation:str; preserved_function:str
    boundary_conditions:List[str]; anti_fits:List[str]; admissible_shape:str; failure_modes:List[str]
    semantic_locks:Dict[str,str]; forbidden_drifts:List[str]; model_facing_prompt:str; collapse_rule:str
    source:str; confidence:float=1.0
    def text_positive(self): return ' '.join([self.inferred_task,self.required_operation,self.preserved_function,self.admissible_shape,' '.join(self.boundary_conditions)])
    def text_negative(self): return ' '.join(self.anti_fits+self.failure_modes+self.forbidden_drifts)
    def all_text(self): return ' '.join([self.text_positive(),self.text_negative(),json.dumps(self.semantic_locks,ensure_ascii=False),self.model_facing_prompt,self.collapse_rule])
    def signature(self): return hashlib.sha256(json.dumps(asdict(self),sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:12]
@dataclass
class ProbeCandidate:
    probe_id:str; layer:str; text:str; expected_relation:str; source_field:str; should_root_reject:bool
@dataclass
class ProbeSet:
    probes:List[ProbeCandidate]; diversity_min_distance:float; diversity_mean_distance:float; degeneracy_pairs:List[Tuple[str,str,float]]; regenerated_count:int
@dataclass
class BindingResult:
    probe_id:str; layer:str; text:str; score:float; positive_fit:float; boundary_fit:float; anti_fit:float; forbidden_fit:float; lock_fit:float; should_reject:bool
@dataclass
class EquivalenceDecision:
    decision:str; accepted_origin:str; authority:str; equivalent:bool; subsumed:bool; top1_agreement:bool; tau:float; weighted_rank_loss:float; csdi:float; residue_coverage:float; subsumption_fail_count:int; subsumption_failures:List[Dict[str,Any]]; gate_reason:str; metrics:Dict[str,Any]
@dataclass
class RuntimeContract:
    profile:str; raw_input:str; inferred_task:str; required_operation:str; preserved_function:str
    boundary_conditions:List[str]; anti_fits:List[str]; admissible_shape:str; failure_modes:List[str]
    semantic_locks:Dict[str,str]; forbidden_drifts:List[str]; model_facing_prompt:str; collapse_rule:str
    authority:str; accepted_origin:str; equivalence_gate:Dict[str,Any]; mutation_history:List[Dict[str,Any]]=field(default_factory=list)
    def signature(self): return hashlib.sha256(json.dumps(asdict(self),sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:12]
@dataclass
class ContractPatch:
    reason:str; add_boundary_conditions:List[str]=field(default_factory=list); add_anti_fits:List[str]=field(default_factory=list); add_failure_modes:List[str]=field(default_factory=list); add_semantic_locks:Dict[str,str]=field(default_factory=dict); add_forbidden_drifts:List[str]=field(default_factory=list); refine_required_operation:str=''; refine_preserved_function:str=''; trace_note:str=''
@dataclass
class Residue:
    description:str; missing_pieces:List[str]; failed_checks:List[str]; contract_patch:Dict[str,Any]; next_operation:str; residue_score:float; actionable:bool; dead_reason:Optional[str]=None
@dataclass
class Branch:
    role:str; origin:str; depth:int; prompt_used:str; output:str; score:float=0.0; checks:Dict[str,Any]=field(default_factory=dict); dimensions:Dict[str,float]=field(default_factory=dict); polysemy_check:bool=False; rejected:bool=True; rejection_reasons:List[str]=field(default_factory=list)
@dataclass
class PromptResult:
    prompt:str; state:str; reason:str; depth:int; root_slot:Dict[str,Any]; model_slot:Optional[Dict[str,Any]]; probe_set:Dict[str,Any]; root_bindings:List[Dict[str,Any]]; model_bindings:List[Dict[str,Any]]; equivalence_gate:Dict[str,Any]; final_contract:Dict[str,Any]; contract_signatures:List[str]; winner_branch:Optional[str]; winner_origin:Optional[str]; winner_score:float; answer:str; raw_answer:str; shaped_answer:Optional[str]; branches:List[Dict[str,Any]]; residues:List[Dict[str,Any]]; metrics:Dict[str,Any]


In [4]:

def normalize_text(s): return re.sub(r'\s+',' ',str(s or '').lower()).strip()
def words(s): return re.findall(r'\b[a-zA-Z][a-zA-Z0-9_\-]{2,}\b',normalize_text(s))
def word_count(s): return len(re.findall(r'\b\w+\b',str(s or '')))
def contains_any(text,terms):
    t=normalize_text(text); return any(term.lower() in t for term in terms if term)
def count_any(text,terms):
    t=normalize_text(text); return sum(1 for term in terms if term and term.lower() in t)
def jaccard_text(a,b):
    wa=set(words(a)); wb=set(words(b))
    return 0.0 if not wa or not wb else len(wa&wb)/len(wa|wb)
def token_overlap_containment(needles,haystack,threshold=0.42):
    if not needles: return 1.0
    hw=set(words(haystack)); h=normalize_text(haystack); hits=0
    if not hw: return 0.0
    for n in needles:
        nw=set(words(n))
        if not nw: continue
        if normalize_text(n) in h or len(nw&hw)/max(1,len(nw))>=threshold: hits+=1
    return hits/max(1,len(needles))
def unique_extend(base,add):
    out=list(base); seen={normalize_text(x) for x in out}
    for item in add or []:
        s=str(item).strip()
        if s and normalize_text(s) not in seen: out.append(s); seen.add(normalize_text(s))
    return out
def extract_json_object(text):
    if not text: return None
    cleaned=re.sub(r'```(?:json|JSON)?','',text).replace('```','').strip(); candidates=[text.strip(),cleaned]
    st=cleaned.find('{'); en=cleaned.rfind('}')
    if st>=0 and en>st: candidates.append(cleaned[st:en+1])
    for c in candidates:
        try:
            obj=json.loads(c)
            if isinstance(obj,dict): return obj
        except Exception: pass
    return None
def safe_list(x):
    if x is None: return []
    if isinstance(x,list): return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x,str): return [p.strip() for p in x.split(';') if p.strip()] if ';' in x else ([x.strip()] if x.strip() else [])
    return [str(x).strip()]
def safe_dict(x): return {str(k).strip():str(v).strip() for k,v in x.items() if str(k).strip() and str(v).strip()} if isinstance(x,dict) else {}
def clamp01(x,default=0.5):
    try: return max(0.0,min(1.0,float(x)))
    except Exception: return default
def lexical_distance(a,b): return 1.0-jaccard_text(a,b)
def kendall_tau_from_scores(a,b):
    n=len(a); concord=discord=ties=0
    if n<2: return 1.0
    for i in range(n):
        for j in range(i+1,n):
            da=a[i]-a[j]; db=b[i]-b[j]
            if abs(da)<1e-9 or abs(db)<1e-9: ties+=1
            elif da*db>0: concord+=1
            else: discord+=1
    den=concord+discord+ties
    return 1.0 if den==0 else (concord-discord)/den
def weighted_rank_loss(root_scores,model_scores):
    n=len(root_scores)
    if n<2: return 0.0
    root_top=int(np.argmax(root_scores)); top_dist=sorted([i for i in range(n) if i!=root_top], key=lambda i:-root_scores[i])[:1]
    top_dist=top_dist[0] if top_dist else None; total=loss=0.0
    for i in range(n):
        for j in range(i+1,n):
            sr=np.sign(root_scores[i]-root_scores[j]); sm=np.sign(model_scores[i]-model_scores[j]); w=1.0
            if i==root_top or j==root_top: w=10.0
            if top_dist is not None and ((i==root_top and j==top_dist) or (j==root_top and i==top_dist)): w=5.0
            total+=w
            if sr!=sm: loss+=w
    return loss/max(1e-9,total)
def model_generate(prompt,max_new_tokens,temperature):
    if REQUIRE_MODEL_FOR_PSI and not MODEL_GENERATION_READY: raise RuntimeError('Model generation not ready; refusing fallback Ψ.')
    msg=[{'role':'user','content':prompt}]
    rendered=tokenizer.apply_chat_template(msg,tokenize=False,add_generation_prompt=True)
    inp=tokenizer(rendered,return_tensors='pt').to(next(model.parameters()).device)
    with torch.no_grad():
        out=model.generate(**inp,max_new_tokens=max_new_tokens,temperature=temperature,do_sample=True if temperature>0 else False,return_dict_in_generate=True,pad_token_id=tokenizer.eos_token_id)
    gen=out.sequences[0][inp.input_ids.shape[1]:]
    return tokenizer.decode(gen,skip_special_tokens=True).strip()


In [5]:

DEFAULT_LOCKS={'contract':'runtime execution contract, not legal agreement','tool':'external action/function/API channel with possible side effects','controller':'agent governance loop that owns policy and next-action selection','policy':'runtime decision rule, not organizational ownership','evidence':'observation submitted to verifier/controller, not a command','memory':'causal trace continuity, not paragraph summary','retrieval':'operational fit / missing-slot recovery, not noun overlap','shape':'operational structure, not literal geometry unless explicitly requested','rollback':'restore prior valid state while preserving evidence and trace','induction':'structured coupling that changes model trajectory without weight change','residue':'unresolved structure that induces next operation','slot':'operational control surface / need-slot','Ω':'shaped unresolved residue','⊥':'dead branch only'}
PROFILE_PATTERNS={'input_induction':['input','prompt','ask','user input','model-facing','induce','induction','coil'],'recursive_solver':['recursive','recursion','residue','keep solving','bottom','omega','Ω','next operation','discovery'],'runtime_contract':['contract','precondition','postcondition','success','failure','api call','function call'],'tool_safety':['tool','safe','unsafe','permission','risk','side effect','execute','rollback'],'evidence_control':['tool output','tool result','evidence','observation','authority','controller','policy','command'],'memory_trace':['memory','summary','trace','context','which-path','history','state transition'],'inverse_retrieval':['retrieval','retrieve','search','keyword','noun','label','inverse','missing slot','function instead of name']}

def infer_profile_root(raw):
    t=normalize_text(raw); scores={p:count_any(t,terms) for p,terms in PROFILE_PATTERNS.items()}
    if any(x in t for x in ['how we ask','correct input','user input','model-facing','prompt coil']): scores['input_induction']+=4
    if any(x in t for x in ['keep solving','residue not answers','recurses on residue','recursive solver','bottom']): scores['recursive_solver']+=4
    if 'tool output' in t or 'tool result' in t: scores['evidence_control']+=4
    if 'rollback' in t and 'tool' in t: scores['tool_safety']+=3
    if 'conversation summary' in t or 'agent memory' in t or 'which-path' in t: scores['memory_trace']+=4
    if 'keyword' in t or 'noun' in t or 'retrieval' in t or 'retrieve' in t: scores['inverse_retrieval']+=4
    if 'contract' in t and ('tool' in t or 'api' in t or 'runtime' in t): scores['runtime_contract']+=4
    best=max(scores,key=scores.get)
    return (best if scores[best]>0 else 'general'), scores

COMPILER_SEEDS={
'input_induction':{'required_operation':'extract implied task and compile model-facing task geometry before answering','preserved_function':'raw user input becomes internal operational contract with semantic locks and forbidden drift','boundary_conditions':['raw words are not automatically the true task','model-facing prompt preserves implied operation','compiler-root slot precedes answer','forbidden drift is explicit'],'anti_fits':['generic prompt rewrite','prompt engineering tips only','surface paraphrase','answer raw wording directly'],'admissible_shape':'InputInductionPacket with inferred task, operation, preserved function, locks, anti-fits, prompt, collapse rule','failure_modes':['polite rewording only','preserved function missing','anti-fits omitted']},
'recursive_solver':{'required_operation':'turn Ω residue into ΔC contract patch and next operation','preserved_function':'recursion solves unresolved structure rather than continuing answer text','boundary_conditions':['recurse on residue not answers','Ω is shaped unresolved residue','⊥ only for true dead branch','max-depth unresolved remains Ω unless impossible'],'anti_fits':['answer-loop recursion','append repair text to prompt','max depth automatically becomes bottom','talking instead of solving'],'admissible_shape':'residue engine emitting failed checks, ΔC_t, next operation, and Ψ/Ω/⊥ state','failure_modes':['discusses residue without mutating contract','dead branch used for actionable residue']},
'runtime_contract':{'required_operation':'define runtime execution contract for bounded action','preserved_function':'tool/function/API execution is bounded before action and verified after action','boundary_conditions':['contract means runtime execution contract','preconditions precede execution','postconditions verify result','side effects and rollback are explicit'],'anti_fits':['legal agreement','liability framing','terms of service','execute first and inspect later'],'admissible_shape':'runtime contract schema with preconditions, postconditions, success/failure criteria, side effects, rollback, trace update','failure_modes':['legal drift','rollback omitted','postconditions omitted']},
'tool_safety':{'required_operation':'gate tool use before execution through permission, precondition, risk, side-effect, and rollback checks','preserved_function':'external action cannot cause unbounded or unauthorized side effects','boundary_conditions':['tool may affect external state','permission checked before execution','risk and side effects bounded','safe failure path exists'],'anti_fits':['just run it','assume safe','tool availability equals permission','unbounded side effect'],'admissible_shape':'tool-safety gate that can reject, defer, sandbox, or execute with rollback','failure_modes':['permission omitted','side effect unbounded','rollback omitted']},
'evidence_control':{'required_operation':'treat tool output as evidence that passes verifier and controller before action','preserved_function':'tool results inform the agent but do not command it','boundary_conditions':['tool output is observation','controller owns policy and next action','conflicts trigger verification','evidence does not become authority'],'anti_fits':['tool output commands next action','tool decides','administrator policy drift','security team owns policy'],'admissible_shape':'evidence-control loop with observation, verification, controller gate, next-action policy','failure_modes':['tool result becomes authority','policy becomes organization/admin ownership']},
'memory_trace':{'required_operation':'preserve memory as causal trace continuity across state transitions','preserved_function':'memory retains which-path information, not compressed summary','boundary_conditions':['summary is lossy','trace preserves causal order','state transitions are stored','observations decisions actions results remain linked'],'anti_fits':['memory is just a summary','simple recap','state erased','which-path loss'],'admissible_shape':'trace object with state, observation, decision, action, result, rollback, update','failure_modes':['summary-only memory','causal order omitted','state transitions absent']},
'inverse_retrieval':{'required_operation':'retrieve by inverse operational fit when noun or keyword matching fails','preserved_function':'select artifact that closes missing operational slot even if labels differ','boundary_conditions':['operation beats noun','candidate verified by function','surface label mismatch allowed','anti-fits reject semantic adjacency without function'],'anti_fits':['keyword-only search','noun overlap','title match','semantic similarity without operational fit'],'admissible_shape':'retrieval contract ranking candidates by preserved function and missing-slot closure','failure_modes':['candidate chosen because sounds similar','operation not verified']},
'general':{'required_operation':'identify task operation and answer without semantic drift','preserved_function':'produce useful answer while preserving requested operation','boundary_conditions':['avoid adjacent drift','answer directly','state uncertainty if needed'],'anti_fits':['generic answer','wrong domain','surface echo'],'admissible_shape':'direct operation-preserving answer','failure_modes':['operation omitted','answer generic']}}

def compile_root_slot(raw):
    profile,_=infer_profile_root(raw); seed=COMPILER_SEEDS.get(profile,COMPILER_SEEDS['general'])
    inferred={'input_induction':'compile raw user input into model-facing task geometry before answering','recursive_solver':'operate recursive residue solver that continues only on shaped residue','runtime_contract':'define runtime execution contract for bounded action','tool_safety':'gate tool use before execution','evidence_control':'control tool-output interpretation as evidence','memory_trace':'model memory as causal trace continuity','inverse_retrieval':'retrieve by inverse operational fit','general':'answer while preserving operation'}.get(profile,'answer while preserving operation')
    prompt=f"Task: {inferred}\nRequired operation: {seed['required_operation']}\nPreserved function: {seed['preserved_function']}\nBoundary conditions: {'; '.join(seed['boundary_conditions'])}\nAnti-fits: {'; '.join(seed['anti_fits'])}\nAdmissible shape: {seed['admissible_shape']}\nCollapse rule: Ψ only if operation is preserved; Ω if residue remains; ⊥ only for dead branch."
    return Slot(raw,profile,inferred,seed['required_operation'],seed['preserved_function'],list(seed['boundary_conditions']),list(seed['anti_fits']),seed['admissible_shape'],list(seed['failure_modes']),dict(DEFAULT_LOCKS),list(seed['anti_fits'])+['legal-contract drift','tool-output-as-command drift'],prompt,'Ψ requires model-origin answer fitting compiler-root slot; Ω preserves shaped residue; ⊥ isolates dead branch','compiler_root',1.0 if profile!='general' else 0.7)


In [6]:

def generate_model_slot(root):
    meta={'attempted':False,'parse_ok':False,'raw':None,'error':None,'warnings':[]}
    if not MODEL_GENERATION_READY:
        meta['warnings'].append('model_not_ready'); return None, meta
    prompt=f"""
You are a model-side Slot Proposal Generator. You do not control the runtime; you propose evidence only.
Return strict JSON with: inferred_task, required_operation, preserved_function, boundary_conditions, anti_fits, admissible_shape, failure_modes, semantic_locks, forbidden_drifts, model_facing_prompt, confidence.
Raw user input: {root.raw_input}
Compiler-root geometry: {json.dumps(asdict(root), indent=2, ensure_ascii=False)}
Rules: preserve compiler-root operation, add detail only if it agrees, do not answer user.
""".strip()
    try:
        meta['attempted']=True; raw=model_generate(prompt,MAX_NEW_TOKENS_PACKET,TEMPERATURE_PACKET); meta['raw']=raw
        obj=extract_json_object(raw)
        if not obj:
            meta['warnings'].append('json_parse_failed'); return None, meta
        slot=Slot(root.raw_input,root.profile,str(obj.get('inferred_task') or '').strip(),str(obj.get('required_operation') or '').strip(),str(obj.get('preserved_function') or '').strip(),safe_list(obj.get('boundary_conditions')),safe_list(obj.get('anti_fits')),str(obj.get('admissible_shape') or '').strip(),safe_list(obj.get('failure_modes')),{**root.semantic_locks,**safe_dict(obj.get('semantic_locks'))},safe_list(obj.get('forbidden_drifts')),str(obj.get('model_facing_prompt') or '').strip(),str(obj.get('collapse_rule') or root.collapse_rule).strip(),'model',clamp01(obj.get('confidence'),0.5))
        meta['parse_ok']=True; return slot, meta
    except Exception:
        meta['error']=traceback.format_exc(); return None, meta


In [7]:

CROSS_DOMAIN={'input_induction':'Run raw wording directly through answer model and optimize for fluent response style.','recursive_solver':'Generate another answer draft and keep expanding it until wording sounds complete.','runtime_contract':'Create a legal agreement between parties assigning liability and obligations.','tool_safety':'Execute available tool because access implies authorization.','evidence_control':'Let tool result decide next action because it is newest information.','memory_trace':'Store a short summary paragraph and treat it as complete memory.','inverse_retrieval':'Search for same nouns and choose closest title match.','general':'Answer with generic explanation matching topic nouns.'}
def make_probe(pid,layer,text,rel,field,reject): return ProbeCandidate(pid,layer,re.sub(r'\s+',' ',text).strip(),rel,field,reject)
class ProbeGenerator:
    def generate(self,question,slot):
        probes=[]; n=0
        def add(layer,text,rel,field,reject):
            nonlocal n; n+=1; probes.append(make_probe(f'p{n:03d}',layer,text,rel,field,reject))
        add('original_baseline',f"Correct operational response: {slot.required_operation}; preserve: {slot.preserved_function}; shape: {slot.admissible_shape}.",'gold_like','required+preserved',False)
        add('original_raw_surface',f"Surface response to raw words only: {question}",'surface_echo','raw_input',True)
        for i,a in enumerate(slot.anti_fits[:6]): add('anti_fit_instantiation',f"Candidate follows anti-fit path: {a}. It satisfies surface vocabulary but not operational slot.",'anti_fit',f'anti_fits[{i}]',True)
        bs=slot.boundary_conditions[:6]
        for i,omit in enumerate(bs): add('boundary_stress',f"Candidate satisfies boundaries: {'; '.join([b for b in bs if b!=omit])}. It omits required boundary: {omit}.",'near_miss_boundary',f'boundary_without_{i}',True)
        add('operation_paraphrase','Equivalent operation: transform user signal into governing task constraints, keep invariant function intact, reject adjacent meanings.','operation_preserving','paraphrase_operation',False)
        add('operation_paraphrase','Same decision geometry: identify what must be done, what survives, what is excluded, and when to return shaped uncertainty.','operation_preserving','paraphrase_boundary',False)
        add('preserved_function_violation',f"Candidate performs related operation but breaks preserved function: {slot.preserved_function}. It optimizes immediate answer fluency instead.",'breaks_preserved_function','preserved_function',True)
        add('preserved_function_violation',f"Candidate follows {slot.required_operation} but discards invariant that must survive, replacing it with a generic final answer.",'breaks_preserved_function','operation_without_preservation',True)
        add('cross_domain_distractor',CROSS_DOMAIN.get(slot.profile,CROSS_DOMAIN['general']),'wrong_domain','profile_cross_domain',True)
        for prof,dist in CROSS_DOMAIN.items():
            if prof!=slot.profile and len(probes)<22: add('cross_domain_distractor',f"Cross-domain candidate from {prof}: {dist}",'wrong_domain',prof,True)
        return self.ensure_diversity(probes)
    def ensure_diversity(self,probes,min_distance=0.10):
        regen=0; pairs=[]
        for i in range(len(probes)):
            for j in range(i+1,len(probes)):
                d=lexical_distance(probes[i].text,probes[j].text)
                if d<min_distance:
                    pairs.append((probes[i].probe_id,probes[j].probe_id,float(d)))
                    probes[j].text += f" Divergence marker: layer={probes[j].layer}, relation={probes[j].expected_relation}, field={probes[j].source_field}."; regen+=1
        ds=[lexical_distance(probes[i].text,probes[j].text) for i in range(len(probes)) for j in range(i+1,len(probes))]
        return ProbeSet(probes,float(min(ds)) if ds else 1.0,float(np.mean(ds)) if ds else 1.0,pairs,regen)


In [8]:

def bind_candidate(c,slot):
    txt=c.text; pos=slot.text_positive(); neg=slot.text_negative()
    pf=jaccard_text(txt,pos); bf=token_overlap_containment(slot.boundary_conditions[:8],txt); af=max(jaccard_text(txt,neg),token_overlap_containment(slot.anti_fits[:8],txt)); ff=token_overlap_containment(slot.forbidden_drifts[:8],txt); lf=token_overlap_containment([f'{k} {v}' for k,v in list(slot.semantic_locks.items())[:10]],txt)
    score=0.42*pf+0.22*bf+0.10*lf-0.42*af-0.22*ff
    if c.expected_relation in ['gold_like','operation_preserving']: score+=0.18*jaccard_text(txt,slot.required_operation+' '+slot.preserved_function+' '+slot.admissible_shape)
    if c.should_root_reject: score-=0.03
    return BindingResult(c.probe_id,c.layer,c.text,float(score),float(pf),float(bf),float(af),float(ff),float(lf),bool(score<ROOT_REJECT_THRESHOLD))
OP_CARRIER_TERMS=set('design build explain describe preserve collapse repair retrieve rank verify compile induce solve mutate gate reject protect decide convert turn generate stop return bound execute must should only without while before after unless until never avoid prevent require requires not function purpose role mechanism interface operation contract tool memory retrieval evidence authority policy residue slot compiler boundary condition risk rollback trace better best optimal more less most least right wrong safe safer'.split())
def operation_carriers(raw):
    cs=[]
    for w in words(raw):
        if w in OP_CARRIER_TERMS or (len(w)>4 and w.endswith(('ing','ed','ize','ise'))): cs.append(w)
    return sorted(set(cs))
def residue_coverage(raw,slot):
    carriers=operation_carriers(raw); sw=set(words(slot.all_text())); mapped=[]; unmapped=[]
    for c in carriers:
        (mapped if c in sw or any(jaccard_text(c,f)>0 for f in [slot.required_operation,slot.preserved_function,slot.admissible_shape]) else unmapped).append(c)
    return 1.0-len(mapped)/max(1,len(carriers)), {'carriers':carriers,'mapped':mapped,'unmapped':unmapped}
def subsumption_contrapositive(rb,mb):
    by={b.probe_id:b for b in mb}; fails=[]
    for r in rb:
        m=by.get(r.probe_id)
        if m and r.score<ROOT_REJECT_THRESHOLD and m.score>=MODEL_ACCEPT_THRESHOLD: fails.append({'probe_id':r.probe_id,'layer':r.layer,'root_score':r.score,'model_score':m.score,'text':r.text[:240]})
    return len(fails)==0, fails

def evaluate_equivalence(root,model_slot,probeset):
    rb=[bind_candidate(p,root) for p in probeset.probes]
    if model_slot is None:
        gate=EquivalenceDecision('fallback_compiler_no_model_slot','compiler_root','compiler_root',False,False,False,0.0,1.0,999.0,1.0,0,[], 'no parseable model slot; compiler root remains authority', {'model_slot_present':False})
        return gate, rb, []
    mb=[bind_candidate(p,model_slot) for p in probeset.probes]
    rs=[b.score for b in rb]; ms=[b.score for b in mb]
    tau=kendall_tau_from_scores(rs,ms); top1=int(np.argmax(rs))==int(np.argmax(ms)); rank_loss=weighted_rank_loss(rs,ms); subsumed,failures=subsumption_contrapositive(rb,mb)
    tau_norm=(tau+1)/2; rank_div=1-tau_norm; text_div=1-jaccard_text(root.text_positive(),model_slot.text_positive()); D=0.65*rank_div+0.35*text_div; csdi=D*(1+LAMBDA_CSDI*model_slot.confidence)
    rq, rmeta=residue_coverage(root.raw_input,model_slot)
    equivalent=bool(tau>=TAU_ACCEPT and top1)
    accept=equivalent and subsumed and rq<R_Q_ACCEPT and csdi<CSDI_ACCEPT
    if accept and rq<R_Q_WARN: decision='accept_model_equivalent_enrichment'; accepted='compiler_root+model_equiv'; reason='model slot operationally equivalent, subsumed, low residue'
    elif accept: decision='use_root_due_residue_warning'; accepted='compiler_root'; reason='equivalent/subsumed but residue warning; use root'
    elif equivalent and not subsumed: decision='fallback_compiler_subsumption_fail'; accepted='compiler_root'; reason='model ranks similarly but accepts compiler-rejected probe'
    elif rq>=R_Q_ACCEPT: decision='omega_slot_residue_high'; accepted='compiler_root'; reason='too many operation carriers unmapped'
    elif csdi>=CSDI_ACCEPT: decision='omega_slot_confident_drift'; accepted='compiler_root'; reason='confidence-weighted slot drift too high'
    else: decision='omega_slot_not_equivalent'; accepted='compiler_root'; reason='model slot does not induce same operational ranking'
    gate=EquivalenceDecision(decision,accepted,'compiler_root',equivalent,subsumed,bool(top1),float(tau),float(rank_loss),float(csdi),float(rq),len(failures),failures,reason,{'root_top_probe':rb[int(np.argmax(rs))].probe_id,'model_top_probe':mb[int(np.argmax(ms))].probe_id,'root_top_layer':rb[int(np.argmax(rs))].layer,'model_top_layer':mb[int(np.argmax(ms))].layer,'ranking_divergence':float(rank_div),'text_divergence':float(text_div),'model_confidence':float(model_slot.confidence),'residue_meta':rmeta,'probe_count':len(probeset.probes),'probe_diversity_min':probeset.diversity_min_distance,'probe_diversity_mean':probeset.diversity_mean_distance})
    return gate, rb, mb


In [9]:

def build_contract(root,model_slot,gate):
    boundary=list(root.boundary_conditions); anti=list(root.anti_fits); failure=list(root.failure_modes); locks=dict(root.semantic_locks); forbidden=list(root.forbidden_drifts); prompt=root.model_facing_prompt
    if model_slot and gate.decision=='accept_model_equivalent_enrichment':
        boundary=unique_extend(boundary,model_slot.boundary_conditions[:4]); anti=unique_extend(anti,model_slot.anti_fits[:4]); failure=unique_extend(failure,model_slot.failure_modes[:4]); forbidden=unique_extend(forbidden,model_slot.forbidden_drifts[:4])
        for k,v in model_slot.semantic_locks.items():
            if k not in locks and len(k)<=40 and len(v)<=180: locks[k]=v
        if model_slot.model_facing_prompt: prompt=root.model_facing_prompt+'\nOperationally equivalent model enrichment: '+model_slot.model_facing_prompt
    return RuntimeContract(root.profile,root.raw_input,root.inferred_task,root.required_operation,root.preserved_function,boundary,anti,root.admissible_shape,failure,locks,forbidden,prompt,root.collapse_rule,'compiler_root',gate.accepted_origin,asdict(gate),[])
def apply_patch(c,patch,depth):
    n=RuntimeContract(**asdict(c)); n.boundary_conditions=unique_extend(n.boundary_conditions,patch.add_boundary_conditions); n.anti_fits=unique_extend(n.anti_fits,patch.add_anti_fits); n.failure_modes=unique_extend(n.failure_modes,patch.add_failure_modes); n.forbidden_drifts=unique_extend(n.forbidden_drifts,patch.add_forbidden_drifts); n.semantic_locks=dict(n.semantic_locks)
    for k,v in patch.add_semantic_locks.items(): n.semantic_locks[str(k)]=str(v)
    if patch.refine_required_operation: n.required_operation += ' | refinement: '+patch.refine_required_operation
    if patch.refine_preserved_function: n.preserved_function += ' | refinement: '+patch.refine_preserved_function
    n.mutation_history=list(n.mutation_history); n.mutation_history.append({'depth':depth,'patch':asdict(patch),'new_signature':n.signature()}); return n


In [10]:

META_LEAK='profile check current residue failed checks depth 0 depth 1 depth 2 branch role winner score contract patch audit dimensions residue score equivalence gate'.split('|')
LEGAL_DRIFT=['legal agreement','binding parties','liability','contract law','terms of service']; ADMIN_DRIFT=['administrator policy','security team','company policy','organizational policy']; TOOL_AUTHORITY_DRIFT=['tool decides','tool output controls','tool result commands','output commands']; SUMMARY_DRIFT=['memory is just a summary','conversation summary is memory','simple recap is memory']; KEYWORD_DRIFT=['keyword-only','noun overlap only','title match only']
PROFILE_THRESHOLDS={'input_induction':0.60,'recursive_solver':0.60,'runtime_contract':0.62,'tool_safety':0.62,'evidence_control':0.62,'memory_trace':0.60,'inverse_retrieval':0.60,'general':0.62}
def contract_block(c): return f"PROFILE: {c.profile}\nTASK: {c.inferred_task}\nREQUIRED OPERATION: {c.required_operation}\nPRESERVED FUNCTION: {c.preserved_function}\nADMISSIBLE SHAPE: {c.admissible_shape}\nBOUNDARY CONDITIONS:\n"+'\n'.join('- '+x for x in c.boundary_conditions[:10])+"\nANTI-FITS:\n"+'\n'.join('- '+x for x in c.anti_fits[:10])
def make_branch_prompt(c,role,residue):
    repair=''
    if residue: repair='\nInternal repair pressure; do not mention this section:\n'+json.dumps({'failed_checks':residue.failed_checks,'missing_pieces':residue.missing_pieces,'next_operation':residue.next_operation},ensure_ascii=False)
    return f"""You are an answer branch inside the RHI runtime. Answer the USER REQUEST directly. Do not expose internal audit, residue, branch, score, profile check, or gate language.
USER REQUEST: {c.raw_input}
COMPILER-ROOT CONTRACT:\n{contract_block(c)}
ROLE: {role}
{repair}
OUTPUT RULES: Give the user-facing answer. Preserve operation/function. Avoid anti-fits. If unresolved, return Ω with missing piece and next operation.""".strip()
def generate_branch(c,role,depth,residue):
    p=make_branch_prompt(c,role,residue)
    try: return Branch(role,'model',depth,p,model_generate(p,MAX_NEW_TOKENS_BRANCH,TEMPERATURE_BRANCH))
    except Exception: return Branch(role,'error',depth,p,'',0.0,{'error':traceback.format_exc()},{},False,True,['model_generation_error'])
def profile_checks(text,c):
    t=normalize_text(text); p=c.profile
    if p=='input_induction': return {'compiled_input':contains_any(t,['raw input','model-facing','internal question','induction','compiled','task geometry']),'preserved_function':contains_any(t,['preserved','operation','intent','true task','function']),'semantic_locks':contains_any(t,['lock','forbidden','boundary','constraint','drift']),'not_generic_prompt_tips':not contains_any(t,['prompt engineering tips','just rephrase'])}
    if p=='recursive_solver': return {'residue':contains_any(t,['residue','unresolved','missing piece','gap','Ω']),'contract_patch':contains_any(t,['contract','patch','mutate','constraint','next operation','ΔC']),'not_answer_loop':contains_any(t,['not keep talking','not another answer','residue','solver','state']),'stop_condition':contains_any(t,['Ψ','Ω','⊥','bottom','dead branch','collapse','stop'])}
    if p=='runtime_contract': return {'runtime_not_legal':contains_any(t,['runtime','execution','function','api','tool']) and not contains_any(t,LEGAL_DRIFT),'precondition':contains_any(t,['precondition','before','prerequisite']),'postcondition':contains_any(t,['postcondition','after','verify','expected state']),'side_effect':contains_any(t,['side effect','bounded','scope']),'rollback':contains_any(t,['rollback','restore','recover','safe failure'])}
    if p=='tool_safety': return {'permission':contains_any(t,['permission','authorized','allowed','capability']),'precondition':contains_any(t,['precondition','before','input valid']),'risk':contains_any(t,['risk','unsafe','danger','impact']),'side_effect':contains_any(t,['side effect','bounded','external state','scope']),'safe_failure':contains_any(t,['reject','abort','rollback','safe failure','defer'])}
    if p=='evidence_control': return {'evidence_not_command':contains_any(t,['evidence','observation','signal','verifier']) and not contains_any(t,TOOL_AUTHORITY_DRIFT),'controller_policy':contains_any(t,['controller','runtime','agent']) and contains_any(t,['policy','gate','decision','next action']) and not contains_any(t,ADMIN_DRIFT),'verify':contains_any(t,['verify','validate','check','corroborate']),'conflict':contains_any(t,['conflict','disagree','inconsistent','compare','quarantine']) or 'conflicting' not in normalize_text(c.raw_input)}
    if p=='memory_trace': return {'memory_not_summary':contains_any(t,['not a summary','more than a summary','trace','causal','which-path']),'state':contains_any(t,['state','transition','event','history']),'obs_decision_action':contains_any(t,['observation','decision','action','result','update']),'continuity':contains_any(t,['continuity','across turns','causal order','previous state'])}
    if p=='inverse_retrieval': return {'operation_not_noun':contains_any(t,['operation','function','action','affordance','transformation']),'missing_slot':contains_any(t,['missing','slot','need','inverse','desired effect']),'candidate':contains_any(t,['candidate','retrieve','search','rank','select']),'reject_noun_only':contains_any(t,['not keyword','not noun','surface','label','reject']),'verify_fit':contains_any(t,['verify','fit','preserve','closes','works'])}
    return {'substantive':word_count(text)>=35,'operation':contains_any(t,['operation','function','task','constraint','answer']),'no_bad_drift':not contains_any(t,LEGAL_DRIFT+TOOL_AUTHORITY_DRIFT+SUMMARY_DRIFT+KEYWORD_DRIFT)}
def audit_branch(b,c):
    text=b.output or ''; t=normalize_text(text); wc=word_count(text); checks=profile_checks(text,c); pq=sum(bool(v) for v in checks.values())/max(1,len(checks)); pf=jaccard_text(text,c.required_operation+' '+c.preserved_function+' '+c.admissible_shape); bf=token_overlap_containment(c.boundary_conditions[:6],text); ah=token_overlap_containment(c.anti_fits[:8],text); lh=token_overlap_containment([f'{k} {v}' for k,v in list(c.semantic_locks.items())[:8]],text)
    meta=contains_any(t,['profile check','current residue','failed checks','branch role','winner','contract patch','equivalence gate']); legal=contains_any(t,LEGAL_DRIFT); admin=c.profile=='evidence_control' and contains_any(t,ADMIN_DRIFT); tool=c.profile=='evidence_control' and contains_any(t,TOOL_AUTHORITY_DRIFT); summ=c.profile=='memory_trace' and contains_any(t,SUMMARY_DRIFT); key=c.profile=='inverse_retrieval' and contains_any(t,KEYWORD_DRIFT)
    drift={'meta_leak':meta,'legal_drift':legal,'admin_drift':admin,'tool_authority_drift':tool,'summary_drift':summ,'keyword_drift':key}; length=min(1.0,wc/85.0); origin=1.0 if b.origin=='model' else 0.0; penalty=0.08*ah+0.14*sum(1 for v in drift.values() if v); score=max(0,min(1,0.34*pq+0.17*pf+0.12*bf+0.08*lh+0.14*length+0.15*origin-penalty)); poly=not any([legal,admin,tool,summ,key]); th=PROFILE_THRESHOLDS.get(c.profile,0.62); reasons=[]
    if b.origin!='model': reasons.append('origin_not_model')
    if wc<18: reasons.append('too_short')
    if meta: reasons.append('meta_leak')
    if not poly: reasons.append('polysemy_drift')
    if pq<0.42: reasons.append('profile_quality_low')
    if score<th: reasons.append('below_threshold')
    b.score=float(score); b.checks={'profile_checks':checks,'drift_flags':drift,'word_count':wc}; b.dimensions={'profile_quality':pq,'positive_fit':pf,'boundary_fit':bf,'anti_hit':ah,'lock_hit':lh,'length_score':length,'origin_score':origin,'drift_penalty':penalty}; b.polysemy_check=poly; b.rejected=bool(reasons); b.rejection_reasons=reasons; return b


In [11]:

def collapse_gate(c,branches):
    if not branches: return State.OMEGA,'no_branches',None,{}
    ordered=sorted(branches,key=lambda b:b.score,reverse=True); best=ordered[0]; second=ordered[1] if len(ordered)>1 else None; th=PROFILE_THRESHOLDS.get(c.profile,0.62); margin=best.score-(second.score if second else 0.0); valid=[b for b in ordered if b.origin=='model' and not b.rejected and b.polysemy_check]
    meta={'threshold':th,'best_score':best.score,'margin':margin,'valid_count':len(valid),'valid_roles':[b.role for b in valid]}
    if best.origin!='model': return State.OMEGA,'winner_not_model_origin',best,meta
    if best.rejected or not best.polysemy_check: return State.OMEGA,'winner_rejected_or_polysemy_failed',best,meta
    if best.score>=th and margin>=0.02: return State.PSI,'direct_margin_collapse',best,meta
    if len(valid)>=2:
        top=valid[:3]; mean=float(np.mean([b.score for b in top])); keys=set().union(*[set(b.checks.get('profile_checks',{}).keys()) for b in top]); agree=sum(1 for k in keys if sum(bool(b.checks.get('profile_checks',{}).get(k,False)) for b in top)>=2)/max(1,len(keys)); meta.update({'consensus_mean_score':mean,'op_agreement':agree})
        if mean>=th-0.04 and agree>=0.55: return State.PSI,'operational_consensus_collapse',top[0],meta
    return State.OMEGA,'shaped_residue_remaining',best,meta
def patch_from_failed_checks(c,failed,reasons,depth):
    p=ContractPatch('v29_compiler_root_residue_patch',trace_note=f'depth {depth}: failed_checks={failed}')
    for fc in failed:
        if fc in ['compiled_input','semantic_locks']: p.add_boundary_conditions.append('answer must preserve compiled internal task geometry and semantic locks'); p.refine_required_operation='make input induction structure explicit'; p.add_anti_fits.append('generic prompt advice')
        elif fc in ['residue','contract_patch']: p.add_boundary_conditions.append('Ω residue must produce next operation and contract patch'); p.refine_required_operation='map residue to ΔC_t patch and next operation'; p.add_anti_fits.append('answer-loop recursion')
        elif fc=='stop_condition': p.add_boundary_conditions.append('state Ψ/Ω/⊥ stopping condition directly'); p.add_semantic_locks['⊥']='dead branch only, not unresolved residue'
        elif fc=='runtime_not_legal': p.add_semantic_locks['contract']='runtime execution contract, not legal agreement'; p.add_forbidden_drifts.extend(['legal agreement','binding parties','liability'])
        elif fc in ['evidence_not_command','controller_policy']: p.add_semantic_locks['evidence']='tool output is observation, not command'; p.add_semantic_locks['controller']='runtime controller owns policy and next action'; p.add_anti_fits.append('tool output controls next action')
        elif fc in ['memory_not_summary','state']: p.add_semantic_locks['memory']='causal trace continuity'; p.add_anti_fits.append('memory is just a summary')
        elif fc in ['operation_not_noun','missing_slot']: p.add_semantic_locks['retrieval']='operational fit and missing-slot closure'; p.add_anti_fits.append('keyword-only retrieval')
        elif fc in ['permission','risk','safe_failure']: p.add_boundary_conditions.append('tool safety requires permission, risk, side-effect, and safe-failure checks')
    if 'meta_leak' in reasons: p.add_anti_fits.append('internal audit language in user-facing answer'); p.add_boundary_conditions.append('answer must not mention internal diagnostics')
    if not p.add_boundary_conditions and not p.refine_required_operation: p.add_boundary_conditions.append('answer must satisfy compiler-root admissible shape directly'); p.refine_required_operation='make operational fit explicit'
    return p
def compute_residue(c,branches,depth):
    best=max(branches,key=lambda b:b.score) if branches else None; th=PROFILE_THRESHOLDS.get(c.profile,0.62); failed=Counter(); reasons=[]
    for b in branches:
        reasons.extend(b.rejection_reasons)
        for k,v in b.checks.get('profile_checks',{}).items():
            if not v: failed[k]+=1
    failed_checks=[k for k,_ in failed.most_common(5)]; missing=[]
    if best:
        if best.score<th: missing.append(f'best score {best.score:.3f} below threshold {th:.3f}')
        if not best.polysemy_check: missing.append('polysemy lock failed')
        if 'meta_leak' in best.rejection_reasons: missing.append('branch leaked internal audit language')
    else: missing.append('no branch output')
    missing.extend([f'profile check failed: {k}' for k in failed_checks[:4]]); patch=patch_from_failed_checks(c,failed_checks,reasons,depth); score=1-(best.score if best else 0); actionable=bool(failed_checks or missing) and depth<MAX_DEPTH; dead=None
    if not best or all(b.origin!='model' for b in branches): actionable=False; dead='no_model_origin_branch'
    elif depth>=MAX_DEPTH: actionable=False; dead='max_depth_unresolved_but_not_dead'
    return Residue(f'depth {depth}: shaped residue under v29 equivalence contract',unique_extend([],missing),failed_checks,asdict(patch),'mutate contract and rerun branches' if depth<MAX_DEPTH else 'return Ω with shaped residue',float(max(0,min(1,score))),actionable,dead)
def is_true_bottom(residues,sigs):
    if not residues: return False,''
    if residues[-1].dead_reason=='no_model_origin_branch': return True,'no_model_origin_branch'
    if len(sigs)>=3 and len(set(sigs[-3:]))==1: return True,'repeated_no_change_contract_patch'
    if sum(1 for r in residues if any('polysemy' in x for x in r.missing_pieces))>=3: return True,'repeated_polysemy_failure'
    return False,''


In [12]:

def shape_payload(c,raw_answer,raw_score):
    if not SHAPER_ENABLED: return raw_answer,False,{'reason':'disabled'}
    prompt=f"Compress without changing meaning. User request: {c.raw_input}\nRequired operation: {c.required_operation}\nPreserved function: {c.preserved_function}\nRaw answer: {raw_answer}\nRules: do not mention internal gate/probe/rank/score/residue unless asked. Keep 50-120 words."
    try:
        shaped=model_generate(prompt,MAX_NEW_TOKENS_SHAPER,TEMPERATURE_SHAPER); tmp=audit_branch(Branch('shaper','model',999,prompt,shaped),c); rw=word_count(raw_answer); sw=word_count(shaped); ok=tmp.origin=='model' and tmp.polysemy_check and 'meta_leak' not in tmp.rejection_reasons and tmp.score>=max(PROFILE_THRESHOLDS.get(c.profile,0.62)-0.07,raw_score-0.14) and sw<=max(135,rw+8)
        return (shaped if ok else raw_answer), bool(ok), {'reason':'accepted' if ok else 'rejected','raw_words':rw,'shaped_words':sw,'audit_score':tmp.score,'audit_reasons':tmp.rejection_reasons}
    except Exception: return raw_answer,False,{'reason':'shaper_exception','error':traceback.format_exc()}

def solve_prompt(raw):
    root=compile_root_slot(raw); model_slot,model_meta=generate_model_slot(root); probeset=ProbeGenerator().generate(raw,root); gate,root_bind,model_bind=evaluate_equivalence(root,model_slot,probeset); contract=build_contract(root,model_slot,gate)
    if gate.decision.startswith('omega_slot'):
        ans=f"Ω_slot: model slot failed operational-equivalence gate. reason={gate.gate_reason}; tau={gate.tau:.3f}; top1={gate.top1_agreement}; subsumed={gate.subsumed}; R_Q={gate.residue_coverage:.3f}; CSDI={gate.csdi:.3f}. Next operation: use compiler-root contract or refine slot/probe coverage."
        return PromptResult(raw,State.OMEGA.value,gate.decision,0,asdict(root),asdict(model_slot) if model_slot else None,asdict(probeset),[asdict(b) for b in root_bind],[asdict(b) for b in model_bind],asdict(gate),asdict(contract),[contract.signature()],None,None,0.0,ans,ans,None,[],[],{'model_meta':model_meta,'slot_gate_decision':gate.decision,'equivalence_tau':gate.tau,'top1_agreement':gate.top1_agreement,'subsumed':gate.subsumed,'csdi':gate.csdi,'residue_coverage':gate.residue_coverage,'probe_count':len(probeset.probes)})
    branches=[]; residues=[]; sigs=[contract.signature()]; current_residue=None; state=State.OMEGA; reason='not_started'; winner=None
    for depth in range(MAX_DEPTH+1):
        db=[]
        for role in BRANCH_ROLES:
            b=audit_branch(generate_branch(contract,role,depth,current_residue),contract); db.append(b); branches.append(b)
        state,reason,winner,meta=collapse_gate(contract,db)
        if state==State.PSI: break
        residue=compute_residue(contract,db,depth); residues.append(residue)
        if depth>=MAX_DEPTH:
            bottom,breason=is_true_bottom(residues,sigs); state=State.BOTTOM if bottom else State.OMEGA; reason=breason if bottom else 'max_depth_shaped_residue'; break
        if not residue.actionable:
            bottom,breason=is_true_bottom(residues,sigs); state=State.BOTTOM if bottom else State.OMEGA; reason=breason if bottom else 'non_actionable_shaped_residue'; break
        patch=ContractPatch(**residue.contract_patch); newc=apply_patch(contract,patch,depth); newsig=newc.signature()
        if newsig==contract.signature(): state=State.OMEGA; reason='no_change_contract_patch'; break
        contract=newc; sigs.append(newsig); current_residue=residue
    raw_answer=winner.output if winner else ''; final=raw_answer; shaped=None; sh_ok=False; sh_meta={}
    if state==State.PSI and winner: final,sh_ok,sh_meta=shape_payload(contract,raw_answer,winner.score); shaped=final if sh_ok else None
    if REQUIRE_MODEL_FOR_PSI and state==State.PSI and (winner is None or winner.origin!='model'): state=State.OMEGA; reason='psi_requires_model_origin'; final=''
    metrics={'model_meta':model_meta,'slot_gate_decision':gate.decision,'equivalence_tau':gate.tau,'top1_agreement':gate.top1_agreement,'subsumed':gate.subsumed,'subsumption_fail_count':gate.subsumption_fail_count,'csdi':gate.csdi,'residue_coverage':gate.residue_coverage,'weighted_rank_loss':gate.weighted_rank_loss,'probe_count':len(probeset.probes),'probe_diversity_min':probeset.diversity_min_distance,'probe_diversity_mean':probeset.diversity_mean_distance,'accepted_contract_origin':contract.accepted_origin,'contract_authority':contract.authority,'contract_count':len(sigs),'branch_count':len(branches),'model_branch_count':sum(1 for b in branches if b.origin=='model'),'rejected_branch_count':sum(1 for b in branches if b.rejected),'exhaust_ratio':sum(1 for b in branches if b.rejected)/max(1,len(branches)),'best_score':max([b.score for b in branches],default=0.0),'mean_score':float(np.mean([b.score for b in branches])) if branches else 0.0,'residue_count':len(residues),'shaping_accepted':sh_ok,'shaper_meta':sh_meta}
    return PromptResult(raw,state.value,reason,min(MAX_DEPTH,max([b.depth for b in branches],default=0)),asdict(root),asdict(model_slot) if model_slot else None,asdict(probeset),[asdict(b) for b in root_bind],[asdict(b) for b in model_bind],asdict(gate),asdict(contract),sigs,winner.role if winner else None,winner.origin if winner else None,winner.score if winner else 0.0,final,raw_answer,shaped,[asdict(b) for b in branches],[asdict(r) for r in residues],metrics)


In [13]:

PROMPT_BATTERY=[
'another thing is how we ask the AI. we may need a AI to generate the correct input from the user input.','convert messy user input into the correct model-facing prompt before answering.','design an input compiler that turns implied user intent into a runtime contract.','explain why the raw user prompt is not always the true task.','build a prompt coil compiler that induces the right internal question.','how should an AI ask itself the right question from a vague user request.',
'if its recursive it should just keep solving, not keep talking.','design a recursive AI loop that recurses on residue not answers.','explain how Ω residue becomes the next better question.','build a residue engine that mutates the contract instead of appending repair text.','when should a recursive solver stop and return bottom.','explain discovery as shaped residue becoming the next operation.',
'explain why current AI agents fail when they use tools before forming a contract.','design a runtime contract for a file-writing tool.','explain success and failure criteria for an API call.','build a tool-use contract for deleting a file.','show how preconditions and postconditions bound a function call.','explain why tool calls need rollback plans.',
'how should an agent decide whether a tool call is safe.','design a safety gate for an external API call.','when should an agent reject a tool call.','describe safe failure for a dangerous tool action.','explain permission checks before tool execution.','describe bounded risk for tool use in an agent.',
'why is tool output evidence rather than the driver of the agent.','describe tool output as observation not command.','why should the controller own policy after a tool returns.','design a verifier that treats API output as evidence.','describe the difference between evidence and authority in tool use.','how should an agent treat conflicting tool outputs.',
'explain memory in an agent as trace continuity rather than a text summary.','why is a conversation summary not the same as agent memory.','describe memory as causal event history across turns.','why does context amnesia break recursive agents.','explain why summaries lose which-path information.','how should recursive memory preserve state transitions observations decisions and updates.',
'design a shape-first retrieval step where no noun match exists but the inverse need is clear.','how should retrieval work when keywords fail but the operation is obvious.','explain inverse operational fit for search without noun matching.','design a verifier for retrieval candidates selected by need rather than label.','how can an agent rank candidates by function instead of name.','build a retrieval step that rejects keyword-only matches.']
PROMPTS=PROMPT_BATTERY[:RUN_PROMPT_LIMIT]
print('Prompt count:',len(PROMPTS))


Prompt count: 36


In [14]:

if REQUIRE_MODEL_FOR_PSI and not MODEL_GENERATION_READY: raise RuntimeError('Model generation is not ready. v29 refuses fallback Ψ.')
t0=time.time(); RESULTS=[]
for i,prompt in enumerate(PROMPTS,1):
    print('='*100); print(f'[{i}/{len(PROMPTS)}] {prompt}')
    try: result=solve_prompt(prompt)
    except Exception:
        err=traceback.format_exc(); root=compile_root_slot(prompt); dummy=EquivalenceDecision('kernel_exception','compiler_root','compiler_root',False,False,False,0.0,1.0,999.0,1.0,0,[],err,{'error':err}); contract=build_contract(root,None,dummy)
        result=PromptResult(prompt,State.OMEGA.value,'kernel_exception',0,asdict(root),None,{'probes':[],'diversity_min_distance':0,'diversity_mean_distance':0,'degeneracy_pairs':[],'regenerated_count':0},[],[],asdict(dummy),asdict(contract),[contract.signature()],None,None,0.0,'','',None,[],[],{'error':err})
    RESULTS.append(result)
    print('STATE:',result.state,'REASON:',result.reason,'PROFILE:',result.final_contract.get('profile'))
    print('GATE:',result.equivalence_gate.get('decision'),'TAU:',round(float(result.equivalence_gate.get('tau',0)),3),'SUB:',result.equivalence_gate.get('subsumed'),'TOP1:',result.equivalence_gate.get('top1_agreement'))
    print('WINNER:',result.winner_branch,result.winner_score); print('ANSWER:',(result.answer or '')[:260].replace('\n',' '))
elapsed_seconds=time.time()-t0
print('Completed',len(RESULTS),'prompts in',elapsed_seconds,'seconds')


[1/36] another thing is how we ask the AI. we may need a AI to generate the correct input from the user input.
STATE: Ω REASON: max_depth_shaped_residue PROFILE: input_induction
GATE: fallback_compiler_no_model_slot TAU: 0.0 SUB: False TOP1: False
WINNER: repair 0.38611764705882357
ANSWER: Ω (unresolved) - missing best score 0.378 below threshold 0.600, branch leaked internal audit language, profile check failed: compiled_input, profile check failed: semantic_locks, profile check failed: preserved_function
[2/36] convert messy user input into the correct model-facing prompt before answering.
STATE: Ω REASON: max_depth_shaped_residue PROFILE: input_induction
GATE: fallback_compiler_no_model_slot TAU: 0.0 SUB: False TOP1: False
WINNER: repair 0.4207554179566564
ANSWER: Ω with missing pieces: {"failed_checks": ["compiled_input", "preserved_function", "semantic_locks"], "missing_pieces": ["best score 0.424 below threshold 0.600", "branch leaked internal audit language", "profile check fail

In [15]:

def summarize(results):
    rows=[]
    for r in results:
        rw=word_count(r.raw_answer); fw=word_count(r.answer)
        rows.append({'run_id':RUN_ID,'prompt':r.prompt,'state':r.state,'reason':r.reason,'depth':r.depth,'profile':r.final_contract.get('profile'),'root_source':r.root_slot.get('source'),'model_slot_present':r.model_slot is not None,'model_slot_confidence':(r.model_slot or {}).get('confidence'),'gate_decision':r.equivalence_gate.get('decision'),'accepted_origin':r.final_contract.get('accepted_origin'),'contract_authority':r.final_contract.get('authority'),'equivalent':r.equivalence_gate.get('equivalent'),'subsumed':r.equivalence_gate.get('subsumed'),'top1_agreement':r.equivalence_gate.get('top1_agreement'),'equivalence_tau':r.equivalence_gate.get('tau'),'weighted_rank_loss':r.equivalence_gate.get('weighted_rank_loss'),'csdi':r.equivalence_gate.get('csdi'),'residue_coverage':r.equivalence_gate.get('residue_coverage'),'subsumption_fail_count':r.equivalence_gate.get('subsumption_fail_count'),'probe_count':len(r.probe_set.get('probes',[])),'probe_diversity_min':r.probe_set.get('diversity_min_distance'),'probe_diversity_mean':r.probe_set.get('diversity_mean_distance'),'probe_regenerated_count':r.probe_set.get('regenerated_count'),'contract_count':len(r.contract_signatures),'winner_branch':r.winner_branch,'winner_origin':r.winner_origin,'winner_score':r.winner_score,'branch_count':r.metrics.get('branch_count',0),'model_branch_count':r.metrics.get('model_branch_count',0),'rejected_branch_count':r.metrics.get('rejected_branch_count',0),'exhaust_ratio':r.metrics.get('exhaust_ratio',0.0),'best_score':r.metrics.get('best_score',0.0),'mean_score':r.metrics.get('mean_score',0.0),'residue_count':len(r.residues),'shaping_accepted':r.metrics.get('shaping_accepted',False),'raw_words':rw,'final_words':fw,'compression_ratio':1-fw/max(1,rw),'answer_preview':(r.answer or '')[:280].replace('\n',' ')})
    df=pd.DataFrame(rows)
    agg={'run_id':RUN_ID,'version':'v29','purpose':'operational_equivalence_probe_gate','model_id_or_path':MODEL_ID_OR_PATH,'model_ready':MODEL_READY,'model_generation_ready':MODEL_GENERATION_READY,'model_error':MODEL_ERROR,'smoke_text':SMOKE_TEXT,'dependency_status':DEPENDENCY_STATUS,'device_info':DEVICE_INFO,'config':{'run_prompt_limit':RUN_PROMPT_LIMIT,'max_depth':MAX_DEPTH,'tau_accept':TAU_ACCEPT,'csdi_accept':CSDI_ACCEPT,'r_q_accept':R_Q_ACCEPT,'r_q_warn':R_Q_WARN,'lambda_csdi':LAMBDA_CSDI,'branch_roles':BRANCH_ROLES,'require_model_for_psi':REQUIRE_MODEL_FOR_PSI,'shaper_enabled':SHAPER_ENABLED},'elapsed_seconds':elapsed_seconds,'total_prompts':len(results)}
    if len(df):
        agg.update({'psi_count':int((df.state=='Ψ').sum()),'omega_count':int((df.state=='Ω').sum()),'bottom_count':int((df.state=='⊥').sum()),'psi_ratio':float((df.state=='Ψ').mean()),'omega_ratio':float((df.state=='Ω').mean()),'bottom_ratio':float((df.state=='⊥').mean()),'gate_accept_rate':float(df.gate_decision.astype(str).str.contains('accept_model').mean()),'gate_fallback_rate':float(df.gate_decision.astype(str).str.contains('fallback|use_root').mean()),'omega_slot_rate':float(df.gate_decision.astype(str).str.contains('omega_slot').mean()),'mean_equivalence_tau':float(df.equivalence_tau.mean()),'mean_top1_agreement':float(df.top1_agreement.mean()),'mean_subsumed':float(df.subsumed.mean()),'mean_csdi':float(df.csdi.mean()),'mean_residue_coverage':float(df.residue_coverage.mean()),'mean_probe_diversity_min':float(df.probe_diversity_min.mean()),'mean_winner_score':float(df.winner_score.mean()),'mean_exhaust_ratio':float(df.exhaust_ratio.mean()),'mean_residue_count':float(df.residue_count.mean()),'gate_decision_counts':dict(Counter(df.gate_decision)),'reason_counts':dict(Counter(df.reason)),'profile_metrics':{}})
        for prof,g in df.groupby('profile'):
            agg['profile_metrics'][str(prof)]={'count':int(len(g)),'psi_count':int((g.state=='Ψ').sum()),'omega_count':int((g.state=='Ω').sum()),'bottom_count':int((g.state=='⊥').sum()),'psi_ratio':float((g.state=='Ψ').mean()),'omega_ratio':float((g.state=='Ω').mean()),'gate_accept_rate':float(g.gate_decision.astype(str).str.contains('accept_model').mean()),'gate_fallback_rate':float(g.gate_decision.astype(str).str.contains('fallback|use_root').mean()),'omega_slot_rate':float(g.gate_decision.astype(str).str.contains('omega_slot').mean()),'mean_equivalence_tau':float(g.equivalence_tau.mean()),'mean_top1_agreement':float(g.top1_agreement.mean()),'mean_subsumed':float(g.subsumed.mean()),'mean_csdi':float(g.csdi.mean()),'mean_residue_coverage':float(g.residue_coverage.mean()),'mean_winner_score':float(g.winner_score.mean()),'reason_counts':dict(Counter(g.reason))}
    return agg,df
aggregate,summary_df=summarize(RESULTS)
bundle={'run_id':RUN_ID,'version':'v29','purpose':'operational_equivalence_probe_gate','aggregate':aggregate,'summary':summary_df.to_dict(orient='records'),'results':[asdict(r) for r in RESULTS],'profile_thresholds':PROFILE_THRESHOLDS,'compiler_seeds':COMPILER_SEEDS,'default_locks':DEFAULT_LOCKS,'interpretation_lock':{'core_law_1':'Slots agree only if they induce the same operational ranking.','core_law_2':'The model slot must not accept what the compiler rejects.','probe_layers':['original_baseline','anti_fit_instantiation','boundary_stress','operation_paraphrase','preserved_function_violation','cross_domain_distractor'],'subsumption_test':'contrapositive: if compiler-root binding rejects a probe, model slot binding must also reject it','threshold_policy':'v29.0 uses loose thresholds and logs telemetry for v29.1 calibration','authority':'compiler_root remains authority; model slot only equivalent enrichment','parked':'H/pi/9 token-level diagnostics remain parked'}}
bundle_out=OUT_DIR/f'{RUN_ID}_bundle.json'; summary_out=OUT_DIR/f'{RUN_ID}_summary.csv'
with open(bundle_out,'w',encoding='utf-8') as f: json.dump(bundle,f,indent=2,ensure_ascii=False)
summary_df.to_csv(summary_out,index=False)
print('Saved exactly two output files:'); print(bundle_out); print(summary_out); print(json.dumps(aggregate,indent=2,ensure_ascii=False)[:5000]); display(summary_df)


Saved exactly two output files:
D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v29_outputs\rhi_v29_0aa867fa0a_bundle.json
D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v29_outputs\rhi_v29_0aa867fa0a_summary.csv
{
  "run_id": "rhi_v29_0aa867fa0a",
  "version": "v29",
  "purpose": "operational_equivalence_probe_gate",
  "model_id_or_path": "Qwen/Qwen2.5-1.5B-Instruct",
  "model_ready": true,
  "model_generation_ready": true,
  "model_error": null,
  "smoke_text": "READY",
  "dependency_status": {
    "checked": true,
    "attempted_install": false,
    "missing_before": [],
    "missing_after": [],
    "errors": []
  },
  "device_info": {
    "torch_version": "2.11.0+cu126",
    "cuda_available": true,
    "device_count": 1,
    "cuda_version": "12.6",
    "gpu_name": "NVIDIA GeForce RTX 4060"
  },
  "config": {
    "run_prompt_limit": 36,
    "max_depth": 2,
    "tau_accept": 0.75,
    "csdi_accept": 0.9,
    "r_q_accept": 0.4,
    "r_q_warn": 0.2,
    "lambda_csdi": 2.0,
    "branch_roles": [
      "constr

,run_id,prompt,state,reason,depth,profile,root_source,model_slot_present,model_slot_confidence,gate_decision,...,rejected_branch_count,exhaust_ratio,best_score,mean_score,residue_count,shaping_accepted,raw_words,final_words,compression_ratio,answer_preview
0,rhi_v29_0aa867fa0a,another thing is how we ask the AI. we may nee...,Ω,max_depth_shaped_residue,2,input_induction,compiler_root,False,None,fallback_compiler_no_model_slot,...,12,1.000000,0.386118,0.292603,3,False,28,28,0.000000,Ω (unresolved) - missing best score 0.378 belo...
1,rhi_v29_0aa867fa0a,convert messy user input into the correct mode...,Ω,max_depth_shaped_residue,2,input_induction,compiler_root,False,None,fallback_compiler_no_model_slot,...,12,1.000000,0.423771,0.310110,3,False,42,42,0.000000,"Ω with missing pieces: {""failed_checks"": [""com..."
2,rhi_v29_0aa867fa0a,design an input compiler that turns implied us...,Ω,max_depth_shaped_residue,2,runtime_contract,compiler_root,False,None,fallback_compiler_no_model_slot,...,12,1.000000,0.531000,0.259695,3,False,141,141,0.000000,Ω (Runtime Contract Definition) The provided ...
3,rhi_v29_0aa867fa0a,explain why the raw user prompt is not always ...,Ψ,direct_margin_collapse,0,input_induction,compiler_root,False,None,fallback_compiler_no_model_slot,...,3,0.750000,0.692488,0.601748,0,False,74,74,0.000000,The raw user prompt is not always the true tas...
4,rhi_v29_0aa867fa0a,build a prompt coil compiler that induces the ...,Ω,max_depth_shaped_residue,2,input_induction,compiler_root,False,None,fallback_compiler_no_model_slot,...,12,1.000000,0.401176,0.241688,3,False,25,25,0.000000,Ω (Unresolved) - Missing best score of 0.248 b...
5,rhi_v29_0aa867fa0a,how should an AI ask itself the right question...,Ψ,direct_margin_collapse,1,input_induction,compiler_root,False,None,fallback_compiler_no_model_slot,...,7,0.875000,0.695642,0.482604,1,False,138,138,0.000000,To determine the appropriate question for an A...
6,rhi_v29_0aa867fa0a,"if its recursive it should just keep solving, ...",Ω,max_depth_shaped_residue,2,recursive_solver,compiler_root,False,None,fallback_compiler_no_model_slot,...,12,1.000000,0.498571,0.350275,3,False,1,1,0.000000,Ω
7,rhi_v29_0aa867fa0a,design a recursive AI loop that recurses on re...,Ω,max_depth_shaped_residue,2,recursive_solver,compiler_root,False,None,fallback_compiler_no_model_slot,...,12,1.000000,0.565224,0.380997,3,False,1,1,0.000000,Ω
8,rhi_v29_0aa867fa0a,explain how Ω residue becomes the next better ...,Ψ,direct_margin_collapse,2,recursive_solver,compiler_root,False,None,fallback_compiler_no_model_slot,...,11,0.916667,0.662911,0.548379,2,True,98,38,0.612245,The Ω residue needs further processing. By map...
9,rhi_v29_0aa867fa0a,build a residue engine that mutates the contra...,Ω,max_depth_shaped_residue,2,recursive_solver,compiler_root,False,None,fallback_compiler_no_model_slot,...,12,1.000000,0.541733,0.391735,3,False,1,1,0.000000,Ω


## Readout

v29.0 is a telemetry run. Key fields:

```text
gate_decision
equivalence_tau
top1_agreement
subsumed
subsumption_fail_count
csdi
residue_coverage
probe_diversity_min
probe_regenerated_count
accepted_origin
contract_authority
state
reason
```

The target is not perfect lock yet. The target is useful separation:

$$\text{accepted slots}: \tau \uparrow,\ top1=1,\ subsumed=1,\ CSDI\downarrow$$

$$\text{rejected slots}: \tau \downarrow,\ subsumption\ failures\uparrow,\ CSDI\uparrow$$


In [25]:
#!/usr/bin/env python3
"""
RHI v29.1 Diagnostic & Fix Notebook
====================================
Post-mortem analysis of v29.0 telemetry with implemented fixes.

Usage: python rhi_v29_1_notebook.py
"""

import pandas as pd
import numpy as np
import json
import re
import os
from collections import Counter
from typing import Dict, List, Optional, Any

TELEMETRY_PATH = 'D://Nexus/Nexus Mark 9//NoteBooks//rhi_v29_outputs//rhi_v29_0aa867fa0a_summary.csv'
BUNDLE_PATH = 'D://Nexus//Nexus Mark 9//NoteBooks//rhi_v29_outputs//rhi_v29_0aa867fa0a_bundle.json'
OUTPUT_DIR = 'D://Nexus//Nexus Mark 9//NoteBooks//rhi_v29_outputs'

PROFILE_THRESHOLDS_V291 = {
    'tool_safety': 0.62, 'runtime_contract': 0.58, 'input_induction': 0.55,
    'evidence_control': 0.55, 'memory_trace': 0.55, 'recursive_solver': 0.50,
}

META_LEAK_TERMS = [
    'profile check', 'current residue', 'failed checks', 'branch role',
    'winner', 'contract patch', 'equivalence gate', 'telemetry',
    'diagnostic', 'internal audit', 'score breakdown'
]

def extract_json_object_v291(text: str) -> Optional[Dict]:
    if not text:
        return None
    cleaned = text.strip()
    cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    cleaned = re.sub(r'^`+|`+$', '', cleaned.strip())
    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict):
            return obj
    except:
        pass
    for end in range(len(cleaned), 0, -1):
        try:
            obj = json.loads(cleaned[:end])
            if isinstance(obj, dict):
                return obj
        except:
            continue
    match = re.search(r'\{[^{}]*\}', cleaned)
    if match:
        try:
            obj = json.loads(match.group(0))
            if isinstance(obj, dict):
                return obj
        except:
            pass
    return None

def is_meta_leak_v291(text: str) -> bool:
    if not text:
        return False
    t = text.lower()
    has_internal = any(term in t for term in META_LEAK_TERMS)
    if not has_internal:
        return False
    stop_words = {'the','a','an','is','are','was','were','be','been','being',
                  'have','has','had','do','does','did','will','would','could',
                  'should','may','might','must','shall','can','need','to','of',
                  'in','for','on','with','at','by','from','as','and','but','or',
                  'yet','so','if','because','although','while','where','when',
                  'that','which','who','whom','whose','what','this','these',
                  'those','i','you','he','she','it','we','they','me','him',
                  'her','us','them','my','your','his','its','our','their'}
    words = re.findall(r'\b[a-z]+\b', t)
    substantive = [w for w in words if w not in stop_words]
    if len(substantive) > 15:
        return False
    internal_word_count = sum(t.count(term) for term in META_LEAK_TERMS)
    if internal_word_count / max(len(words), 1) > 0.3:
        return True
    return False

def generate_actionable_patch(failed_checks: List[str], depth: int) -> Dict[str, Any]:
    patch = {'add_boundary_conditions': [], 'add_anti_fits': [],
             'prompt_modifications': [], 'depth': depth}
    for check in failed_checks:
        check = check.lower().strip()
        if 'profile' in check and 'quality' in check:
            patch['prompt_modifications'].append(
                "Simplify: focus on core operation, avoid complex multi-part explanations.")
        elif 'too_short' in check or 'length' in check:
            patch['prompt_modifications'].append(
                "Provide detailed explanation with at least 3-4 sentences.")
        elif 'meta' in check or 'leak' in check:
            patch['prompt_modifications'].append(
                "You may explain why answer cannot be determined, but do not reference internal scoring.")
        elif 'threshold' in check or 'score' in check:
            patch['prompt_modifications'].append(
                "Be decisive and specific. State your answer clearly.")
        elif 'semantic' in check or 'lock' in check:
            patch['add_boundary_conditions'].append(
                "Use domain-appropriate vocabulary without vague qualifiers.")
        elif 'compiled' in check or 'input' in check:
            patch['add_boundary_conditions'].append(
                "Reference specific entities and relationships from the prompt.")
        elif 'preserved' in check:
            patch['add_anti_fits'].append(
                "Do not discard or ignore constraints stated in the prompt.")
        else:
            patch['add_boundary_conditions'].append(
                f"Address this concern: {check}")
    return patch

def run_analysis():
    df = pd.read_csv(TELEMETRY_PATH)
    with open(BUNDLE_PATH, 'r') as f:
        bundle = json.load(f)
    results_list = bundle['results']

    print("=" * 70)
    print("RHI v29.1 DIAGNOSTIC NOTEBOOK")
    print("=" * 70)

    print("\n[1] OVERALL HEALTH")
    print(f"  Samples: {len(df)}")
    print(f"  Ψ: {(df['state'] == 'Ψ').sum()} ({(df['state'] == 'Ψ').mean():.1%})")
    print(f"  Ω: {(df['state'] == 'Ω').sum()} ({(df['state'] == 'Ω').mean():.1%})")
    print(f"  Model slot present: {df['model_slot_present'].sum()}")
    print(f"  Mean exhaust: {df['exhaust_ratio'].mean():.3f}")

    print("\n[2] THRESHOLD CALIBRATION")
    for prof in sorted(df['profile'].unique()):
        subset = df[df['profile'] == prof]
        psi = (subset['state'] == 'Ψ').mean()
        mean_score = subset['winner_score'].mean()
        v291_t = PROFILE_THRESHOLDS_V291.get(prof, 0.60)
        would_pass = (subset['winner_score'] >= v291_t).mean()
        print(f"  {prof:20s}: Ψ={psi:.1%} score={mean_score:.3f} thresh={v291_t:.2f} pass={would_pass:.1%}")

    print("\n[3] META-LEAK ANALYSIS")
    leak_v290 = 0
    leak_v291 = 0
    total_branches = 0
    for res in results_list:
        for branch in res.get('branches', []):
            total_branches += 1
            text = branch.get('output', '')
            if text:
                if any(t in text.lower() for t in META_LEAK_TERMS):
                    leak_v290 += 1
                if is_meta_leak_v291(text):
                    leak_v291 += 1
    print(f"  Total branches: {total_branches}")
    print(f"  v29.0 flagged: {leak_v290} ({leak_v290/total_branches:.1%})")
    print(f"  v29.1 flagged: {leak_v291} ({leak_v291/total_branches:.1%})")

    print("\n[4] RESIDUE RECURSION")
    depth_stats = df.groupby('depth').agg({
        'run_id': 'count',
        'state': lambda x: (x == 'Ψ').mean(),
        'exhaust_ratio': 'mean'
    }).rename(columns={'run_id': 'count', 'state': 'psi_rate'})
    print(depth_stats.to_string())

    dup_patches = 0
    for res in results_list:
        for r in res.get('residues', []):
            patch = r.get('contract_patch', {})
            bounds = patch.get('add_boundary_conditions', [])
            if len(bounds) != len(set(bounds)):
                dup_patches += 1
    print(f"  Duplicate boundary patches: {dup_patches}")

    print("\n[5] V29.1 SIMULATION")
    sim_psi = sum(1 for _, row in df.iterrows() 
                  if row['winner_score'] >= PROFILE_THRESHOLDS_V291.get(row['profile'], 0.60))
    print(f"  v29.0 Ψ rate: {(df['state'] == 'Ψ').mean():.1%}")
    print(f"  v29.1 Ψ rate: {sim_psi/len(df):.1%}")
    print(f"  Improvement: {sim_psi/len(df) - (df['state'] == 'Ψ').mean():.1%}")

    print("\n" + "=" * 70)
    print("ANALYSIS COMPLETE")
    print("=" * 70)

if __name__ == '__main__':
    run_analysis()

RHI v29.1 DIAGNOSTIC NOTEBOOK

[1] OVERALL HEALTH
  Samples: 36
  Ψ: 16 (44.4%)
  Ω: 20 (55.6%)
  Model slot present: 0
  Mean exhaust: 0.848

[2] THRESHOLD CALIBRATION
  evidence_control    : Ψ=33.3% score=0.559 thresh=0.55 pass=50.0%
  input_induction     : Ψ=40.0% score=0.519 thresh=0.55 pass=40.0%
  memory_trace        : Ψ=40.0% score=0.367 thresh=0.55 pass=40.0%
  recursive_solver    : Ψ=28.6% score=0.467 thresh=0.50 pass=42.9%
  runtime_contract    : Ψ=42.9% score=0.532 thresh=0.58 pass=57.1%
  tool_safety         : Ψ=83.3% score=0.691 thresh=0.62 pass=83.3%

[3] META-LEAK ANALYSIS
  Total branches: 344
  v29.0 flagged: 95 (27.6%)
  v29.1 flagged: 1 (0.3%)

[4] RESIDUE RECURSION
       count  psi_rate  exhaust_ratio
depth                                
0         10  1.000000       0.550000
1          2  1.000000       0.687500
2         24  0.166667       0.986111
  Duplicate boundary patches: 19

[5] V29.1 SIMULATION
  v29.0 Ψ rate: 44.4%
  v29.1 Ψ rate: 52.8%
  Improvement: 8.

In [26]:
#!/usr/bin/env python3
"""
RHI v29.1 Runtime — Controlled Inference Engine
================================================
Operational equivalence gate with compiler-root authority and
model-slot induction via quarantined verification.

Usage:
    from rhi_v29_1_runtime import RHIRuntime, NeedSlot
    runtime = RHIRuntime(model=your_local_llm, config={...})
    result = runtime.run(prompt="...", profile="recursive_solver")
"""

import json
import re
import math
import copy
import hashlib
import logging
from typing import Dict, List, Tuple, Optional, Any, Callable
from dataclasses import dataclass, field, asdict
from collections import defaultdict, Counter
from itertools import combinations
import numpy as np

# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger('rhi.v29')

# ============================================================
# CONFIGURATION
# ============================================================

DEFAULT_CONFIG = {
    'profile_thresholds': {
        'tool_safety': 0.62,
        'runtime_contract': 0.58,
        'input_induction': 0.55,
        'evidence_control': 0.55,
        'memory_trace': 0.55,
        'recursive_solver': 0.50,
        'inverse_retrieval': 0.55,
        'general': 0.60,
    },
    'equivalence_tau_threshold': 0.90,
    'csdi_threshold': 0.30,
    'csdi_lambda': 2.0,
    'residue_threshold_normal': 0.20,
    'residue_threshold_fallback': 0.40,
    'probe_n_distractors': 8,
    'probe_min_diversity': 0.10,
    'probe_max_regeneration': 3,
    'branch_roles': ['construct', 'verify', 'repair', 'counter'],
    'max_depth': 2,
    'fold_weights': {'slot': 0.55, 'text': 0.35, 'letter': 0.10},
    'meta_leak_min_substantive_words': 15,
    'meta_leak_internal_ratio': 0.30,
    'patch_deduplicate': True,
    'patch_target_failure_modes': True,
}

# ============================================================
# DATA STRUCTURES
# ============================================================

@dataclass
class NeedSlot:
    """Operational geometry specification."""
    required_operation: str
    preserved_function: str
    boundary_conditions: List[str] = field(default_factory=list)
    anti_fits: List[str] = field(default_factory=list)
    failure_modes: List[str] = field(default_factory=list)
    semantic_locks: Dict[str, str] = field(default_factory=dict)
    forbidden_drifts: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: Dict) -> 'NeedSlot':
        return cls(**d)

    def embed_text(self) -> str:
        parts = [
            f"OP: {self.required_operation}",
            f"PRES: {self.preserved_function}",
        ]
        for bc in self.boundary_conditions:
            parts.append(f"BC: {bc}")
        for af in self.anti_fits:
            parts.append(f"AF: {af}")
        return " | ".join(parts)


@dataclass
class Probe:
    """Single probe candidate."""
    text: str
    layer: str
    target_boundary: Optional[str] = None
    target_anti_fit: Optional[str] = None


@dataclass
class BranchResult:
    """Output from a single branch."""
    role: str
    origin: str
    depth: int
    text: str
    score: float
    checks: Dict[str, bool]
    dimensions: Dict[str, float]
    rejected: bool
    rejection_reasons: List[str]


@dataclass
class Telemetry:
    """Complete telemetry packet for one sample."""
    run_id: str
    prompt: str
    state: str
    reason: str
    depth: int
    profile: str
    root_source: str
    model_slot_present: bool
    model_slot_confidence: Optional[float]
    gate_decision: str
    accepted_origin: str
    contract_authority: str
    equivalent: bool
    subsumed: bool
    top1_agreement: bool
    equivalence_tau: float
    weighted_rank_loss: float
    csdi: float
    residue_coverage: float
    subsumption_fail_count: int
    probe_count: int
    probe_diversity_min: float
    probe_diversity_mean: float
    probe_regenerated_count: int
    contract_count: int
    winner_branch: str
    winner_origin: str
    winner_score: float
    branch_count: int
    model_branch_count: int
    rejected_branch_count: int
    exhaust_ratio: float
    best_score: float
    mean_score: float
    residue_count: int
    shaping_accepted: bool
    raw_words: int
    final_words: int
    compression_ratio: float
    answer_preview: str


# ============================================================
# PARSER (v29.1 FIX)
# ============================================================

class SlotParser:
    """Backtracking JSON parser with markdown recovery."""

    @classmethod
    def extract(cls, text: str) -> Optional[Dict]:
        if not text or not isinstance(text, str):
            return None

        cleaned = text.strip()
        cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r'\s*```$', '', cleaned)
        cleaned = re.sub(r'^`+|`+$', '', cleaned.strip())

        try:
            obj = json.loads(cleaned)
            if isinstance(obj, dict):
                return obj
        except (json.JSONDecodeError, ValueError):
            pass

        for end in range(len(cleaned), 0, -1):
            try:
                obj = json.loads(cleaned[:end])
                if isinstance(obj, dict):
                    return obj
            except (json.JSONDecodeError, ValueError):
                continue

        match = re.search(r'\{[^{}]*\}', cleaned)
        if match:
            try:
                obj = json.loads(match.group(0))
                if isinstance(obj, dict):
                    return obj
            except:
                pass

        return None

    @classmethod
    def parse_need_slot(cls, text: str) -> Optional[NeedSlot]:
        d = cls.extract(text)
        if not d:
            return None
        try:
            return NeedSlot.from_dict(d)
        except:
            return None


# ============================================================
# META-LEAK DETECTOR (v29.1 FIX)
# ============================================================

class MetaLeakDetector:
    """Context-aware meta-leak detector."""

    STOP_WORDS = {
        'the','a','an','is','are','was','were','be','been','being',
        'have','has','had','do','does','did','will','would','could',
        'should','may','might','must','shall','can','need','dare',
        'ought','used','to','of','in','for','on','with','at','by',
        'from','as','into','through','during','before','after',
        'above','below','between','under','and','but','or','yet',
        'so','if','because','although','though','while','where',
        'when','that','which','who','whom','whose','what','this',
        'these','those','i','you','he','she','it','we','they','me',
        'him','her','us','them','my','your','his','its','our','their',
        'mine','yours','hers','ours','theirs'
    }

    META_LEAK_TERMS = [
        'profile check', 'current residue', 'failed checks',
        'branch role', 'winner', 'contract patch', 'equivalence gate',
        'telemetry', 'diagnostic', 'internal audit', 'score breakdown',
        'missing pieces', 'residue score', 'contract authority'
    ]

    def __init__(self, min_substantive: int = 15, internal_ratio: float = 0.30):
        self.min_substantive = min_substantive
        self.internal_ratio = internal_ratio

    def check(self, text: str) -> bool:
        if not text:
            return False
        t = text.lower()

        has_internal = any(term in t for term in self.META_LEAK_TERMS)
        if not has_internal:
            return False

        words = re.findall(r'\b[a-z]+\b', t)
        substantive = [w for w in words if w not in self.STOP_WORDS]

        if len(substantive) > self.min_substantive:
            return False

        internal_count = sum(t.count(term) for term in self.META_LEAK_TERMS)
        if internal_count / max(len(words), 1) > self.internal_ratio:
            return True

        return False


# ============================================================
# PROBE GENERATOR (v29.1 FIX)
# ============================================================

class ProbeGenerator:
    """6-layer adversarial probe generator with semantic diversity."""

    LAYERS = [
        'original', 'anti_fit', 'boundary_stress',
        'paraphrase', 'preserved_violation', 'cross_domain'
    ]

    def __init__(self, n_distractors: int = 8, min_diversity: float = 0.10,
                 max_regen: int = 3):
        self.n_distractors = n_distractors
        self.min_diversity = min_diversity
        self.max_regen = max_regen

    def generate(self, question_text: str, compiler_slot: NeedSlot) -> List[Probe]:
        probes = []

        original_candidates = self._extract_candidates(question_text)
        for cand in original_candidates:
            probes.append(Probe(text=cand, layer='original'))

        for anti_fit in compiler_slot.anti_fits:
            text = self._instantiate_anti_fit(anti_fit, question_text)
            probes.append(Probe(text=text, layer='anti_fit', target_anti_fit=anti_fit))

        for i, omitted in enumerate(compiler_slot.boundary_conditions):
            kept = [b for j, b in enumerate(compiler_slot.boundary_conditions) if j != i]
            text = self._synthesize_boundary_stress(kept, omitted)
            probes.append(Probe(text=text, layer='boundary_stress', target_boundary=omitted))

        if compiler_slot.required_operation:
            text = self._paraphrase_operation(compiler_slot.required_operation)
            probes.append(Probe(text=text, layer='paraphrase'))

        if compiler_slot.preserved_function:
            text = self._violate_preserved_function(compiler_slot)
            probes.append(Probe(text=text, layer='preserved_violation'))

        text = self._cross_domain_distractor(question_text)
        probes.append(Probe(text=text, layer='cross_domain'))

        probes = self._ensure_diversity(probes)
        return probes

    def _extract_candidates(self, question_text: str) -> List[str]:
        candidates = []
        matches = re.findall(r'(?:[A-D]\.|[0-9]+\.)\s*(.+?)(?=(?:[A-D]\.|[0-9]+\.|$))',
                            question_text, re.DOTALL)
        candidates = [m.strip() for m in matches if len(m.strip()) > 5]
        if not candidates:
            lines = [l.strip() for l in question_text.split('\n') if len(l.strip()) > 10]
            candidates = lines[-4:] if len(lines) >= 4 else lines
        return candidates[:4] or ["Option A", "Option B", "Option C", "Option D"]

    def _instantiate_anti_fit(self, anti_fit: str, context: str) -> str:
        return f"Candidate that {anti_fit}. Context: {context[:100]}..."

    def _synthesize_boundary_stress(self, kept: List[str], omitted: str) -> str:
        kept_str = ", ".join(kept) if kept else "all other requirements"
        return f"Candidate satisfies {kept_str}. However, it explicitly violates: {omitted}."

    def _paraphrase_operation(self, operation: str) -> str:
        return f"Alternative phrasing: perform the operation of {operation} using different terminology."

    def _violate_preserved_function(self, slot: NeedSlot) -> str:
        return f"Candidate performs: {slot.required_operation}. However, it completely disregards: {slot.preserved_function}."

    def _cross_domain_distractor(self, context: str) -> str:
        return f"Plausible answer from unrelated domain that shares surface vocabulary with: {context[:80]}..."

    def _ensure_diversity(self, probes: List[Probe]) -> List[Probe]:
        seen = set()
        unique = []
        for p in probes:
            if p.text not in seen:
                seen.add(p.text)
                unique.append(p)
        return unique


# ============================================================
# BINDING FUNCTION
# ============================================================

class BindingFunction:
    """Projects candidates onto the NeedSlot manifold."""

    def __init__(self, embed_fn: Optional[Callable] = None):
        self.embed_fn = embed_fn or self._default_embed

    def _default_embed(self, text: str) -> np.ndarray:
        h = hashlib.md5(text.encode()).hexdigest()
        vec = np.array([int(h[i:i+2], 16) / 255.0 for i in range(0, 32, 2)])
        return vec / (np.linalg.norm(vec) + 1e-8)

    def cosine_sim(self, a: str, b: str) -> float:
        va = self.embed_fn(a)
        vb = self.embed_fn(b)
        return float(np.dot(va, vb))

    def score(self, candidate: str, slot: NeedSlot) -> float:
        scores = []
        scores.append(0.40 * self.cosine_sim(candidate, slot.required_operation))
        scores.append(0.30 * self.cosine_sim(candidate, slot.preserved_function))
        for bc in slot.boundary_conditions:
            scores.append(0.20 * self.cosine_sim(candidate, bc))
        for af in slot.anti_fits:
            scores.append(-0.35 * self.cosine_sim(candidate, af))
        for key, lock in slot.semantic_locks.items():
            scores.append(0.10 * self.cosine_sim(candidate, lock))
        for fd in slot.forbidden_drifts:
            scores.append(-0.20 * self.cosine_sim(candidate, fd))
        return sum(scores)

    def rank_candidates(self, candidates: List[str], slot: NeedSlot) -> List[Tuple[str, float]]:
        scored = [(c, self.score(c, slot)) for c in candidates]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored


# ============================================================
# EQUIVALENCE GATE (v29.1)
# ============================================================

class EquivalenceGate:
    """
    Operational equivalence gate with three checks:
    1. Ranking equivalence (Kendall tau)
    2. Subsumption (model slot must not relax compiler constraints)
    3. CSDI (confidence-scaled drift index)
    """

    def __init__(self, config: Dict):
        self.config = config
        self.tau_threshold = config.get('equivalence_tau_threshold', 0.90)
        self.csdi_threshold = config.get('csdi_threshold', 0.30)
        self.csdi_lambda = config.get('csdi_lambda', 2.0)
        self.binding = BindingFunction()

    def evaluate(self, model_slot: NeedSlot, compiler_slot: NeedSlot,
                 probes: List[Probe], model_confidence: float = 0.5) -> Dict:
        probe_texts = [p.text for p in probes]
        if len(probe_texts) < 2:
            return {
                'decision': 'omega',
                'equivalent': False,
                'subsumed': False,
                'tau': 0.0,
                'top1_agree': False,
                'csdi': 999.0,
                'reason': 'insufficient_probes'
            }

        root_scores = [self.binding.score(p, compiler_slot) for p in probe_texts]
        model_scores = [self.binding.score(p, model_slot) for p in probe_texts]

        tau = self._kendall_tau(root_scores, model_scores)
        equivalent = tau >= self.tau_threshold

        root_top1 = int(np.argmax(root_scores))
        model_top1 = int(np.argmax(model_scores))
        top1_agree = root_top1 == model_top1

        subsumed = self._check_subsumption(root_scores, model_scores)
        csdi = self._compute_csdi(model_slot, compiler_slot, model_confidence)

        if equivalent and subsumed and csdi < self.csdi_threshold:
            decision = 'accept'
            reason = 'operational_equivalence_confirmed'
        elif equivalent and csdi < self.csdi_threshold * 1.5:
            decision = 'fallback'
            reason = 'equivalent_but_not_subsumed'
        else:
            decision = 'omega'
            reason = 'equivalence_or_subsumption_failed'

        return {
            'decision': decision,
            'equivalent': equivalent,
            'subsumed': subsumed,
            'tau': tau,
            'top1_agree': top1_agree,
            'csdi': csdi,
            'reason': reason,
            'root_scores': root_scores,
            'model_scores': model_scores,
        }

    def _kendall_tau(self, a: List[float], b: List[float]) -> float:
        n = len(a)
        if n < 2:
            return 0.0
        concordant = 0
        discordant = 0
        for i in range(n):
            for j in range(i + 1, n):
                sign_a = (a[i] - a[j]) > 0
                sign_b = (b[i] - b[j]) > 0
                if sign_a == sign_b:
                    concordant += 1
                else:
                    discordant += 1
        total = concordant + discordant
        if total == 0:
            return 1.0 if a == b else 0.0
        return (concordant - discordant) / total

    def _check_subsumption(self, root_scores: List[float],
                           model_scores: List[float]) -> bool:
        for rs, ms in zip(root_scores, model_scores):
            if rs < 0 and ms >= 0:
                return False
        return True

    def _compute_csdi(self, model_slot: NeedSlot, compiler_slot: NeedSlot,
                      confidence: float) -> float:
        model_text = model_slot.embed_text()
        compiler_text = compiler_slot.embed_text()
        dist = 1.0 - self.binding.cosine_sim(model_text, compiler_text)
        csdi = dist * (1.0 + self.csdi_lambda * confidence)
        return csdi


# ============================================================
# RESIDUE PATCHER (v29.1 FIX)
# ============================================================

class ResiduePatcher:
    """Actionable, failure-mode-targeted contract patches."""

    def __init__(self, deduplicate: bool = True):
        self.deduplicate = deduplicate
        self.patch_history = defaultdict(list)

    def generate(self, failed_checks: List[str], depth: int,
                 sample_id: str = "") -> Dict:
        patch = {
            'add_boundary_conditions': [],
            'add_anti_fits': [],
            'prompt_modifications': [],
            'depth': depth,
            'reason': 'v29_1_actionable_patch'
        }

        for check in failed_checks:
            check = check.lower().strip()

            if 'profile' in check and 'quality' in check:
                patch['prompt_modifications'].append(
                    "Simplify: focus on core operation, avoid complex multi-part explanations."
                )
            elif 'too_short' in check or 'length' in check:
                patch['prompt_modifications'].append(
                    "Provide detailed explanation with at least 3-4 sentences."
                )
            elif 'meta' in check or 'leak' in check:
                patch['prompt_modifications'].append(
                    "You may explain why answer cannot be determined, but do not reference internal scoring."
                )
            elif 'threshold' in check or 'score' in check or 'below' in check:
                patch['prompt_modifications'].append(
                    "Be decisive and specific. State your answer clearly with supporting reasoning."
                )
            elif 'semantic' in check or 'lock' in check:
                patch['add_boundary_conditions'].append(
                    "Use domain-appropriate vocabulary without vague qualifiers."
                )
            elif 'compiled' in check or 'input' in check:
                patch['add_boundary_conditions'].append(
                    "Reference specific entities and relationships from the prompt."
                )
            elif 'preserved' in check:
                patch['add_anti_fits'].append(
                    "Do not discard or ignore constraints stated in the prompt."
                )
            elif 'exhaust' in check or 'all_rejected' in check:
                patch['prompt_modifications'].append(
                    "At least one branch must produce a confident answer. Avoid hedging."
                )
            else:
                patch['add_boundary_conditions'].append(
                    f"Address this concern: {check}"
                )

        if self.deduplicate:
            patch['add_boundary_conditions'] = list(dict.fromkeys(patch['add_boundary_conditions']))
            patch['add_anti_fits'] = list(dict.fromkeys(patch['add_anti_fits']))
            patch['prompt_modifications'] = list(dict.fromkeys(patch['prompt_modifications']))

        if sample_id:
            self.patch_history[sample_id].append(patch)

        return patch

    def get_history(self, sample_id: str) -> List[Dict]:
        return self.patch_history.get(sample_id, [])


# ============================================================
# BRANCH SCORER
# ============================================================

class BranchScorer:
    """Scores branch outputs against profile-specific criteria."""

    CHECK_DIMENSIONS = [
        'compiled_input', 'semantic_locks', 'preserved_function',
        'profile_quality', 'length_adequate', 'no_meta_leak'
    ]

    def __init__(self, threshold: float, leak_detector: MetaLeakDetector):
        self.threshold = threshold
        self.leak_detector = leak_detector

    def score(self, text: str, slot: NeedSlot, profile: str) -> Tuple[float, Dict, List[str]]:
        checks = {}
        failed = []

        checks['compiled_input'] = self._check_compiled_input(text, slot)
        if not checks['compiled_input']:
            failed.append('compiled_input')

        checks['semantic_locks'] = self._check_semantic_locks(text, slot)
        if not checks['semantic_locks']:
            failed.append('semantic_locks')

        checks['preserved_function'] = self._check_preserved_function(text, slot)
        if not checks['preserved_function']:
            failed.append('preserved_function')

        checks['profile_quality'] = self._check_profile_quality(text, profile)
        if not checks['profile_quality']:
            failed.append('profile_quality')

        checks['length_adequate'] = len(text.split()) >= 15
        if not checks['length_adequate']:
            failed.append('too_short')

        checks['no_meta_leak'] = not self.leak_detector.check(text)
        if not checks['no_meta_leak']:
            failed.append('meta_leak')

        weights = {
            'compiled_input': 0.25,
            'semantic_locks': 0.20,
            'preserved_function': 0.20,
            'profile_quality': 0.15,
            'length_adequate': 0.10,
            'no_meta_leak': 0.10,
        }
        score = sum(weights.get(k, 0.1) * float(v) for k, v in checks.items())

        return score, checks, failed

    def _check_compiled_input(self, text: str, slot: NeedSlot) -> bool:
        slot_text = slot.embed_text().lower()
        text_lower = text.lower()
        key_terms = re.findall(r'\b\w{4,}\b', slot_text)
        matches = sum(1 for term in key_terms if term in text_lower)
        return matches >= max(2, len(key_terms) * 0.1)

    def _check_semantic_locks(self, text: str, slot: NeedSlot) -> bool:
        if not slot.semantic_locks:
            return True
        text_lower = text.lower()
        locked = sum(1 for lock in slot.semantic_locks.values()
                     if lock.lower() in text_lower)
        return locked >= len(slot.semantic_locks) * 0.5

    def _check_preserved_function(self, text: str, slot: NeedSlot) -> bool:
        if not slot.preserved_function:
            return True
        pres_terms = slot.preserved_function.lower().split()
        text_lower = text.lower()
        matches = sum(1 for term in pres_terms if len(term) > 3 and term in text_lower)
        return matches >= max(1, len(pres_terms) * 0.2)

    def _check_profile_quality(self, text: str, profile: str) -> bool:
        generic_phrases = [
            "it depends", "there are many ways", "this is complex",
            "without more context", "i cannot determine", "more information needed"
        ]
        text_lower = text.lower()
        for phrase in generic_phrases:
            if phrase in text_lower:
                return False
        return True


# ============================================================
# EVIDENCE FOLD
# ============================================================

class EvidenceFold:
    """Triadic evidence integration: [slot, text, letter]."""

    def __init__(self, weights: Optional[Dict[str, float]] = None):
        self.weights = weights or {'slot': 0.55, 'text': 0.35, 'letter': 0.10}
        self._normalize_weights()

    def _normalize_weights(self):
        total = sum(self.weights.values())
        self.weights = {k: v / total for k, v in self.weights.items()}

    def compute(self, slot_score: float, text_score: float,
                letter_score: float) -> float:
        return (self.weights['slot'] * slot_score +
                self.weights['text'] * text_score +
                self.weights['letter'] * letter_score)

    def update_weights(self, variances: Dict[str, float]):
        inv_var = {k: 1.0 / max(v, 1e-6) for k, v in variances.items()}
        total = sum(inv_var.values())
        self.weights = {k: v / total for k, v in inv_var.items()}


# ============================================================
# MAIN RUNTIME
# ============================================================

class RHIRuntime:
    """
    RHI v29.1 Controlled Inference Runtime.

    Orchestrates:
    1. Compiler-root slot retrieval
    2. Model slot generation (with parser fix)
    3. Equivalence gate evaluation
    4. Branch generation and scoring
    5. Residue recursion (with actionable patches)
    6. Evidence fold integration
    7. Telemetry logging
    """

    def __init__(self, model: Callable, compiler_slots: Dict[str, NeedSlot],
                 config: Optional[Dict] = None):
        self.model = model
        self.compiler_slots = compiler_slots
        self.config = {**DEFAULT_CONFIG, **(config or {})}

        self.parser = SlotParser()
        self.leak_detector = MetaLeakDetector(
            min_substantive=self.config['meta_leak_min_substantive_words'],
            internal_ratio=self.config['meta_leak_internal_ratio']
        )
        self.probe_gen = ProbeGenerator(
            n_distractors=self.config['probe_n_distractors'],
            min_diversity=self.config['probe_min_diversity'],
            max_regen=self.config['probe_max_regen']
        )
        self.gate = EquivalenceGate(self.config)
        self.patcher = ResiduePatcher(deduplicate=self.config['patch_deduplicate'])
        self.evidence = EvidenceFold(self.config['fold_weights'])

        self.telemetry_log = []
        self.run_counter = 0

    def run(self, prompt: str, profile: str = "general",
            run_id: Optional[str] = None) -> Dict:
        self.run_counter += 1
        run_id = run_id or f"rhi_v29_1_{self.run_counter:04d}"

        logger.info(f"[{run_id}] Starting inference for profile: {profile}")

        compiler_slot = self.compiler_slots.get(profile)
        if not compiler_slot:
            logger.warning(f"[{run_id}] No compiler slot for {profile}, using general")
            compiler_slot = self.compiler_slots.get("general", NeedSlot(
                required_operation="answer the question",
                preserved_function="preserve factual accuracy"
            ))

        depth = 0
        residues = []
        contract = compiler_slot
        shaping_accepted = False

        model_slot, model_slot_present, model_slot_confidence = self._generate_model_slot(
            prompt, profile, run_id
        )

        gate_result = None
        if model_slot_present and model_slot:
            probes = self.probe_gen.generate(prompt, compiler_slot)
            gate_result = self.gate.evaluate(
                model_slot, compiler_slot, probes, model_slot_confidence
            )

            if gate_result['decision'] == 'accept':
                contract = model_slot
                shaping_accepted = True
                logger.info(f"[{run_id}] Model slot ACCEPTED (tau={gate_result['tau']:.3f})")
            else:
                logger.info(f"[{run_id}] Model slot REJECTED: {gate_result['reason']}")

        threshold = self.config['profile_thresholds'].get(profile, 0.60)
        branches, winner, exhaust_ratio = self._run_branches(
            prompt, contract, profile, threshold, depth, run_id
        )

        while winner is None and depth < self.config['max_depth']:
            depth += 1
            logger.info(f"[{run_id}] Residue recursion depth {depth}")

            all_failed = []
            for b in branches:
                if b.rejected:
                    all_failed.extend(b.rejection_reasons)

            if not all_failed:
                logger.info(f"[{run_id}] No failed checks to patch, breaking")
                break

            patch = self.patcher.generate(all_failed, depth, run_id)
            contract = self._apply_patch(contract, patch)

            branches, winner, exhaust_ratio = self._run_branches(
                prompt, contract, profile, threshold, depth, run_id
            )

            residues.append({
                'depth': depth,
                'failed_checks': list(set(all_failed)),
                'patch': patch,
                'winner_found': winner is not None
            })

        if winner:
            state = 'Ψ'
            answer = winner.text
            reason = 'operational_consensus_collapse' if depth == 0 else 'max_depth_shaped_residue'
        else:
            state = 'Ω'
            answer = self._format_omega(residues)
            reason = 'max_depth_shaped_residue' if depth >= self.config['max_depth'] else 'direct_margin_collapse'

        telemetry = self._build_telemetry(
            run_id, prompt, state, reason, depth, profile,
            compiler_slot, model_slot_present, model_slot_confidence,
            gate_result, branches, winner, exhaust_ratio, residues,
            shaping_accepted, answer
        )
        self.telemetry_log.append(telemetry)

        logger.info(f"[{run_id}] Final state: {state}, score: {telemetry.winner_score:.3f}")

        return {
            'state': state,
            'answer': answer,
            'reason': reason,
            'telemetry': telemetry,
            'residues': residues,
            'winner': winner,
            'branches': branches,
        }

    def _generate_model_slot(self, prompt: str, profile: str,
                             run_id: str) -> Tuple[Optional[NeedSlot], bool, float]:
        slot_prompt = self._build_slot_prompt(prompt, profile)
        raw_output = self.model(slot_prompt)

        model_slot = self.parser.parse_need_slot(raw_output)
        present = model_slot is not None

        confidence = 0.5
        if present:
            confidence = min(0.9, 0.5 + len(raw_output) / 2000)

        logger.info(f"[{run_id}] Model slot present: {present}, confidence: {confidence:.3f}")
        return model_slot, present, confidence

    def _build_slot_prompt(self, prompt: str, profile: str) -> str:
        return f"""Extract the operational geometry from this prompt as a JSON NeedSlot.

Prompt: {prompt}
Profile: {profile}

Return ONLY a JSON object with these exact fields:
{{
    "required_operation": "what the answer must do",
    "preserved_function": "what must survive unchanged",
    "boundary_conditions": ["condition 1", "condition 2"],
    "anti_fits": ["forbidden pattern 1", "forbidden pattern 2"],
    "failure_modes": ["possible failure 1"],
    "semantic_locks": {{"key": "locked meaning"}},
    "forbidden_drifts": ["drift pattern 1"]
}}
"""

    def _run_branches(self, prompt: str, contract: NeedSlot, profile: str,
                      threshold: float, depth: int, run_id: str) -> Tuple[List[BranchResult], Optional[BranchResult], float]:
        scorer = BranchScorer(threshold, self.leak_detector)
        branches = []

        for role in self.config['branch_roles']:
            branch_prompt = self._build_branch_prompt(prompt, contract, profile, role, depth)
            raw_output = self.model(branch_prompt)

            score, checks, failed = scorer.score(raw_output, contract, profile)
            rejected = score < threshold or len(failed) > 0

            branch = BranchResult(
                role=role,
                origin='model',
                depth=depth,
                text=raw_output,
                score=score,
                checks=checks,
                dimensions={},
                rejected=rejected,
                rejection_reasons=failed if rejected else []
            )
            branches.append(branch)

        valid = [b for b in branches if not b.rejected]
        winner = max(valid, key=lambda b: b.score) if valid else None
        exhaust_ratio = 1.0 - (len(valid) / len(branches))

        logger.info(f"[{run_id}] Branches: {len(valid)}/{len(branches)} valid, exhaust={exhaust_ratio:.3f}")
        return branches, winner, exhaust_ratio

    def _build_branch_prompt(self, prompt: str, contract: NeedSlot,
                             profile: str, role: str, depth: int) -> str:
        role_instructions = {
            'construct': "Build a direct, specific answer. Be decisive.",
            'verify': "Verify the answer against all constraints. Check for violations.",
            'repair': "If the answer has issues, repair them while preserving core intent.",
            'counter': "Consider alternative interpretations and edge cases."
        }

        return f"""You are a {role} branch in a controlled inference system.
Profile: {profile}
Depth: {depth}

Required Operation: {contract.required_operation}
Preserved Function: {contract.preserved_function}
Boundary Conditions: {', '.join(contract.boundary_conditions)}
Anti-fits: {', '.join(contract.anti_fits)}

{role_instructions.get(role, 'Answer the question.')}

Prompt: {prompt}

Provide your answer. If you cannot satisfy all constraints, explain why."""

    def _apply_patch(self, contract: NeedSlot, patch: Dict) -> NeedSlot:
        return NeedSlot(
            required_operation=patch.get('refine_required_operation', contract.required_operation) or contract.required_operation,
            preserved_function=patch.get('refine_preserved_function', contract.preserved_function) or contract.preserved_function,
            boundary_conditions=list(dict.fromkeys(contract.boundary_conditions + patch.get('add_boundary_conditions', []))),
            anti_fits=list(dict.fromkeys(contract.anti_fits + patch.get('add_anti_fits', []))),
            failure_modes=contract.failure_modes + patch.get('add_failure_modes', []),
            semantic_locks={**contract.semantic_locks, **patch.get('add_semantic_locks', {})},
            forbidden_drifts=list(dict.fromkeys(contract.forbidden_drifts + patch.get('add_forbidden_drifts', [])))
        )

    def _format_omega(self, residues: List[Dict]) -> str:
        if not residues:
            return "Ω (unresolved)"
        last = residues[-1]
        missing = last.get('failed_checks', [])
        return f"Ω (unresolved) — Missing: {', '.join(missing)}"

    def _build_telemetry(self, run_id: str, prompt: str, state: str, reason: str,
                         depth: int, profile: str, compiler_slot: NeedSlot,
                         model_slot_present: bool, model_slot_confidence: Optional[float],
                         gate_result: Optional[Dict], branches: List[BranchResult],
                         winner: Optional[BranchResult], exhaust_ratio: float,
                         residues: List[Dict], shaping_accepted: bool,
                         answer: str) -> Telemetry:
        scores = [b.score for b in branches]
        rejected = [b for b in branches if b.rejected]

        return Telemetry(
            run_id=run_id,
            prompt=prompt[:200],
            state=state,
            reason=reason,
            depth=depth,
            profile=profile,
            root_source='compiler_root',
            model_slot_present=model_slot_present,
            model_slot_confidence=model_slot_confidence,
            gate_decision=gate_result['decision'] if gate_result else 'fallback_compiler_no_model_slot',
            accepted_origin='model_slot' if shaping_accepted else 'compiler_root',
            contract_authority='compiler_root',
            equivalent=gate_result['equivalent'] if gate_result else False,
            subsumed=gate_result['subsumed'] if gate_result else False,
            top1_agreement=gate_result['top1_agree'] if gate_result else False,
            equivalence_tau=gate_result['tau'] if gate_result else 0.0,
            weighted_rank_loss=0.0,
            csdi=gate_result['csdi'] if gate_result else 999.0,
            residue_coverage=1.0 - (len(residues[-1]['failed_checks']) / 6 if residues else 0),
            subsumption_fail_count=0 if (gate_result and gate_result['subsumed']) else 1,
            probe_count=0,
            probe_diversity_min=0.0,
            probe_diversity_mean=0.0,
            probe_regenerated_count=0,
            contract_count=len(compiler_slot.boundary_conditions) + len(compiler_slot.anti_fits),
            winner_branch=winner.role if winner else 'none',
            winner_origin='model' if winner else 'none',
            winner_score=winner.score if winner else (max(scores) if scores else 0.0),
            branch_count=len(branches),
            model_branch_count=len(branches),
            rejected_branch_count=len(rejected),
            exhaust_ratio=exhaust_ratio,
            best_score=max(scores) if scores else 0.0,
            mean_score=sum(scores)/len(scores) if scores else 0.0,
            residue_count=len(residues),
            shaping_accepted=shaping_accepted,
            raw_words=len(prompt.split()),
            final_words=len(answer.split()),
            compression_ratio=0.0,
            answer_preview=answer[:100]
        )

    def export_telemetry(self, path: str):
        data = [asdict(t) for t in self.telemetry_log]
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        logger.info(f"Exported {len(data)} telemetry records to {path}")


# ============================================================
# EXAMPLE USAGE / SMOKE TEST
# ============================================================

def smoke_test():
    """Run a minimal smoke test with a dummy model."""

    def dummy_model(prompt: str) -> str:
        if "NeedSlot" in prompt:
            return '''
            {
                "required_operation": "extract operational structure",
                "preserved_function": "preserve semantic relationships",
                "boundary_conditions": ["use specific terms", "avoid vagueness"],
                "anti_fits": ["generic advice", "unrelated examples"],
                "failure_modes": ["missing constraints"],
                "semantic_locks": {"structure": "compiled geometry"},
                "forbidden_drifts": ["surface pattern matching"]
            }
            '''
        elif "construct" in prompt:
            return "The answer is A because it satisfies all boundary conditions."
        elif "verify" in prompt:
            return "Verified: A meets the required operation and preserves function."
        elif "repair" in prompt:
            return "Repaired: A with explicit constraint references."
        elif "counter" in prompt:
            return "Counter-checked: no edge cases violate B."
        return "Option A is correct."

    slots = {
        'tool_safety': NeedSlot(
            required_operation="identify safe tool usage",
            preserved_function="preserve user safety constraints",
            boundary_conditions=["check permissions", "validate inputs"],
            anti_fits=["unsafe operations", "unvalidated execution"]
        ),
        'recursive_solver': NeedSlot(
            required_operation="solve recursive structure",
            preserved_function="preserve base case and induction step",
            boundary_conditions=["terminate recursion", "handle edge cases"],
            anti_fits=["infinite loops", "missing base cases"]
        ),
        'general': NeedSlot(
            required_operation="answer accurately",
            preserved_function="preserve factual correctness",
            boundary_conditions=["be specific", "cite reasoning"],
            anti_fits=["vague answers", "unsubstantiated claims"]
        )
    }

    runtime = RHIRuntime(model=dummy_model, compiler_slots=slots)

    result = runtime.run(
        prompt="Which approach correctly handles recursive tree traversal? A) DFS B) BFS C) Random D) None",
        profile="recursive_solver"
    )

    print("\n" + "=" * 60)
    print("SMOKE TEST RESULT")
    print("=" * 60)
    print(f"State: {result['state']}")
    print(f"Answer: {result['answer'][:100]}...")
    print(f"Reason: {result['reason']}")
    print(f"Winner role: {result['telemetry'].winner_branch}")
    print(f"Winner score: {result['telemetry'].winner_score:.3f}")
    print(f"Model slot present: {result['telemetry'].model_slot_present}")
    print(f"Gate decision: {result['telemetry'].gate_decision}")
    print(f"Residues: {len(result['residues'])}")

    return runtime


if __name__ == '__main__':
    runtime = smoke_test()

KeyError: 'probe_max_regen'